# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 완료 주문과 취소 주문만 사용해 `is_cancelled`를 예측하는 이진 분류 실습입니다.

## 학습 목표

- `completed=0`, `cancelled=1`로 타깃 범위를 명확하게 정의합니다.
- `refunded`와 기타 상태를 이진 분류 대상에서 제외합니다.
- 병합의 키 관계, 행 수, 미매칭을 검증합니다.
- train, validation, test를 분리합니다.
- Dummy 기준 모델, Logistic Regression, Random Forest를 비교합니다.
- validation에서 모델과 임계값을 선택하고 test는 최종 평가에 한 번만 사용합니다.
- 예측 결과에서 고객명 등 불필요한 개인정보를 제외합니다.


## 실행 조건과 사전 고정 규칙

- 공식 원본의 단계와 질문을 유지하며 내장 코드로 같은 분석을 재현함
- 공통 원본 300주문에서 refunded 52건을 Target 제외하고, 이진 대상 중 가입일 이전 45건은 날짜 의미 미해결로 보류하여 203건을 분석함
- 날짜 보류는 Test 성능을 보기 전에 정했으며 업무 오류를 수정한 것으로 처리하지 않음
- 상세와 결제수단이 확정됐지만 취소 여부는 확정되기 전을 예측 시점으로 가정하며 고객 정보도 해당 시점에 가용해야 함
- 비Dummy Validation F1, Recall, Precision, 모델명 순으로 모델을 선택하고 threshold는 0.20~0.80/0.05 간격에서 F1, Recall, Precision, 낮은 값 순으로 선택함
- Test 확인 전 별도 Validation 실행에서 Logistic Regression과 0.60을 저장했으며 이 Notebook은 같은 선택과 입력 및 분할 해시를 확인한 뒤 고정 평가를 재현함
- 내장 전처리는 전체 고객의 나이 중앙값 대체를 제거하고 Train Pipeline에서만 결측치 대체, 표준화 및 인코딩을 학습함
- 실제 FP/FN 비용이 없어 F1은 교육용 사전 기준이며 운영 사용은 보류함


## 1. 프로젝트 루트와 실행 환경 설정

전처리 파일이 없다면 프로젝트 루트에서 먼저 다음 명령을 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [1]:
from pathlib import Path
import sys, json, tempfile, hashlib, urllib.request, subprocess, os, shutil
import pandas as pd
import numpy as np
from IPython.display import display

REFERENCE_COMMIT = "6105f42bd8294cb364dde6dcc2fee71c32e14794"
RAW_HASHES = {'customers': '3b31aaefa66505f0a1820ccd89f0eac3b5e12830b71ecb45f2b65928bcdfb143', 'products': '46ff06753ade3e71011ee08dd366148dcd9c36521f111840cead0a9d1047e405', 'orders': '8f6e7ee39aa5ee6e229199532caa84bd9d4d30ef2abc2b9412cf45b0b2fdbfae', 'order_items': 'a7a5c3942b0886c66cb38cc6a10fdfa68dbcde90e8642585dee49c4bd104063a'}
EMBEDDED_SOURCES = {'src/__init__.py': '"""LLM 데이터 분석 실습용 공통 모듈."""\n', 'src/data_loader.py': 'from pathlib import Path\n\nimport pandas as pd\n\n\ndef load_csv(file_path: str | Path) -> pd.DataFrame:\n    """CSV 파일을 읽어 pandas DataFrame으로 반환합니다."""\n    path = Path(file_path)\n    if not path.exists():\n        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")\n    return pd.read_csv(path)\n\n\ndef load_sales_data(data_dir: str | Path = "data/raw") -> dict[str, pd.DataFrame]:\n    """실습용 쇼핑몰 CSV 파일 4개를 한 번에 불러옵니다."""\n    base_dir = Path(data_dir)\n    return {\n        "customers": load_csv(base_dir / "customers.csv"),\n        "products": load_csv(base_dir / "products.csv"),\n        "orders": load_csv(base_dir / "orders.csv"),\n        "order_items": load_csv(base_dir / "order_items.csv"),\n    }\n', 'src/preprocessing.py': '"""Chapter 5 데이터 전처리 공통 함수 모음.\n\n이 모듈은 5장 노트북과 스크립트에서 함께 사용할 수 있는 전처리 함수를 제공합니다.\n원본 DataFrame을 직접 수정하지 않고 항상 복사본을 반환하도록 작성했습니다.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport pandas as pd\n\n\nSTATUS_MAP = {\n    "complete": "completed",\n    "Complete": "completed",\n    "COMPLETED": "completed",\n    "완료": "completed",\n    "cancel": "cancelled",\n    "Cancel": "cancelled",\n    "CANCELLED": "cancelled",\n    "취소": "cancelled",\n    "refund": "refunded",\n    "Refund": "refunded",\n    "REFUNDED": "refunded",\n    "환불": "refunded",\n}\n\n\ndef check_missing_values(df: pd.DataFrame) -> pd.Series:\n    """각 컬럼의 결측치 개수를 확인합니다."""\n    return df.isna().sum()\n\n\ndef remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:\n    """중복 행을 제거한 새 DataFrame을 반환합니다."""\n    return df.drop_duplicates().reset_index(drop=True)\n\n\ndef convert_date_column(df: pd.DataFrame, column: str) -> pd.DataFrame:\n    """문자열 날짜 컬럼을 datetime 형식으로 변환합니다."""\n    result = df.copy()\n    result[column] = pd.to_datetime(result[column], errors="coerce")\n    return result\n\n\ndef missing_summary(df: pd.DataFrame) -> pd.DataFrame:\n    """컬럼별 결측치 개수와 비율을 반환합니다."""\n    summary = pd.DataFrame(\n        {\n            "missing_count": df.isna().sum(),\n            "missing_ratio": (df.isna().mean() * 100).round(2),\n        }\n    )\n    return summary.sort_values("missing_count", ascending=False)\n\n\ndef shape_summary(data: dict[str, pd.DataFrame]) -> pd.DataFrame:\n    """여러 DataFrame의 행/열 개수를 요약합니다."""\n    rows = []\n    for name, df in data.items():\n        rows.append({"dataset": name, "rows": df.shape[0], "columns": df.shape[1]})\n    return pd.DataFrame(rows)\n\n\ndef compare_shapes(\n    raw_data: dict[str, pd.DataFrame],\n    processed_data: dict[str, pd.DataFrame],\n) -> pd.DataFrame:\n    """전처리 전후 데이터 크기를 비교합니다."""\n    raw_shapes = shape_summary(raw_data).rename(\n        columns={"rows": "rows_raw", "columns": "columns_raw"}\n    )\n    processed_shapes = shape_summary(processed_data).rename(\n        columns={"rows": "rows_processed", "columns": "columns_processed"}\n    )\n    return raw_shapes.merge(processed_shapes, on="dataset", how="outer")\n\n\ndef duplicate_summary(\n    data: dict[str, pd.DataFrame],\n    key_columns: dict[str, str] | None = None,\n) -> pd.DataFrame:\n    """전체 행 중복과 주요 키 컬럼 중복을 요약합니다."""\n    key_columns = key_columns or {}\n    rows = []\n\n    for name, df in data.items():\n        key_col = key_columns.get(name)\n        key_duplicate_count = None\n        if key_col and key_col in df.columns:\n            key_duplicate_count = int(df[key_col].duplicated().sum())\n\n        rows.append(\n            {\n                "dataset": name,\n                "row_duplicate_count": int(df.duplicated().sum()),\n                "key_column": key_col,\n                "key_duplicate_count": key_duplicate_count,\n            }\n        )\n\n    return pd.DataFrame(rows)\n\n\ndef strip_string_columns(df: pd.DataFrame) -> pd.DataFrame:\n    """문자열 컬럼의 앞뒤 공백을 제거합니다.\n\n    결측치를 문자열 \'nan\'으로 바꾸지 않기 위해 결측치는 그대로 유지합니다.\n    """\n    result = df.copy()\n    string_columns = result.select_dtypes(include="object").columns\n\n    for col in string_columns:\n        result[col] = result[col].where(\n            result[col].isna(),\n            result[col].astype(str).str.strip(),\n        )\n\n    return result\n\n\ndef to_number(series: pd.Series) -> pd.Series:\n    """쉼표가 포함된 문자열 숫자를 숫자형으로 변환합니다."""\n    return pd.to_numeric(\n        series.astype(str).str.replace(",", "", regex=False),\n        errors="coerce",\n    )\n\n\ndef normalize_order_status(series: pd.Series) -> pd.Series:\n    """주문 상태값의 대표 표기를 통일합니다."""\n    return series.replace(STATUS_MAP)\n\n\ndef preprocess_customers(df: pd.DataFrame) -> pd.DataFrame:\n    """고객 데이터 전처리.\n\n    처리 내용:\n    - 문자열 앞뒤 공백 제거\n    - age 숫자형 변환 및 중앙값 대체\n    - city 결측치 Unknown 처리\n    - signup_date 날짜형 변환\n    - 완전 중복 행 제거\n    """\n    result = strip_string_columns(df)\n\n    if "age" in result.columns:\n        result["age"] = pd.to_numeric(result["age"], errors="coerce")\n        # Chapter10: defer imputation statistics to the Train-only Pipeline.\n\n\n    if "signup_date" in result.columns:\n        result["signup_date"] = pd.to_datetime(result["signup_date"], errors="coerce")\n\n    return result.drop_duplicates()\n\n\ndef preprocess_products(df: pd.DataFrame) -> pd.DataFrame:\n    """상품 데이터 전처리.\n\n    처리 내용:\n    - 문자열 앞뒤 공백 제거\n    - price 숫자형 변환\n    - price가 0 이하인 행 제외\n    - 완전 중복 행 제거\n    """\n    result = strip_string_columns(df)\n\n    if "price" in result.columns:\n        result["price"] = to_number(result["price"])\n        result = result[result["price"] > 0]\n\n    return result.drop_duplicates()\n\n\ndef preprocess_orders(df: pd.DataFrame) -> pd.DataFrame:\n    """주문 데이터 전처리.\n\n    처리 내용:\n    - 문자열 앞뒤 공백 제거\n    - order_status 대표 표기 통일\n    - order_date 날짜형 변환\n    - order_month, order_dayofweek 파생 컬럼 생성\n    - 완전 중복 행 제거\n    """\n    result = strip_string_columns(df)\n\n    if "order_status" in result.columns:\n        result["order_status"] = normalize_order_status(result["order_status"])\n\n    if "order_date" in result.columns:\n        result["order_date"] = pd.to_datetime(result["order_date"], errors="coerce")\n        result["order_month"] = result["order_date"].dt.to_period("M").astype(str)\n        result["order_dayofweek"] = result["order_date"].dt.day_name()\n\n    return result.drop_duplicates()\n\n\ndef preprocess_order_items(df: pd.DataFrame) -> pd.DataFrame:\n    """주문 상세 데이터 전처리.\n\n    처리 내용:\n    - 문자열 앞뒤 공백 제거\n    - quantity, unit_price 숫자형 변환\n    - quantity와 unit_price가 0 이하인 행 제외\n    - line_total 파생 컬럼 생성\n    - 완전 중복 행 제거\n    """\n    result = strip_string_columns(df)\n\n    if "quantity" in result.columns:\n        result["quantity"] = to_number(result["quantity"])\n        result = result[result["quantity"] > 0]\n\n    if "unit_price" in result.columns:\n        result["unit_price"] = to_number(result["unit_price"])\n        result = result[result["unit_price"] > 0]\n\n    if {"quantity", "unit_price"}.issubset(result.columns):\n        result["line_total"] = result["quantity"] * result["unit_price"]\n\n    return result.drop_duplicates()\n\n\ndef preprocess_sales_data(data: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:\n    """쇼핑몰 실습 데이터 4종을 한 번에 전처리합니다."""\n    return {\n        "customers": preprocess_customers(data["customers"]),\n        "products": preprocess_products(data["products"]),\n        "orders": preprocess_orders(data["orders"]),\n        "order_items": preprocess_order_items(data["order_items"]),\n    }\n\n\ndef validate_relationships(data: dict[str, pd.DataFrame]) -> pd.DataFrame:\n    """전처리 후 파일 간 키 관계가 유지되는지 확인합니다."""\n    customers = data["customers"]\n    products = data["products"]\n    orders = data["orders"]\n    order_items = data["order_items"]\n\n    checks = []\n\n    if {"customer_id"}.issubset(customers.columns) and {"customer_id"}.issubset(orders.columns):\n        invalid_customers = orders[~orders["customer_id"].isin(customers["customer_id"])]\n        checks.append(\n            {\n                "check": "orders.customer_id exists in customers.customer_id",\n                "invalid_count": len(invalid_customers),\n            }\n        )\n\n    if {"order_id"}.issubset(orders.columns) and {"order_id"}.issubset(order_items.columns):\n        invalid_orders = order_items[~order_items["order_id"].isin(orders["order_id"])]\n        checks.append(\n            {\n                "check": "order_items.order_id exists in orders.order_id",\n                "invalid_count": len(invalid_orders),\n            }\n        )\n\n    if {"product_id"}.issubset(products.columns) and {"product_id"}.issubset(order_items.columns):\n        invalid_products = order_items[\n            ~order_items["product_id"].isin(products["product_id"])\n        ]\n        checks.append(\n            {\n                "check": "order_items.product_id exists in products.product_id",\n                "invalid_count": len(invalid_products),\n            }\n        )\n\n    return pd.DataFrame(checks)\n\n\ndef save_processed_data(\n    data: dict[str, pd.DataFrame],\n    output_dir: str | Path = "data/processed",\n    encoding: str = "utf-8-sig",\n) -> list[Path]:\n    """전처리 결과를 CSV 파일로 저장하고 저장 경로 목록을 반환합니다."""\n    output_path = Path(output_dir)\n    output_path.mkdir(parents=True, exist_ok=True)\n\n    file_map = {\n        "customers": "customers_clean.csv",\n        "products": "products_clean.csv",\n        "orders": "orders_clean.csv",\n        "order_items": "order_items_clean.csv",\n    }\n\n    saved_paths: list[Path] = []\n    for name, filename in file_map.items():\n        path = output_path / filename\n        data[name].to_csv(path, index=False, encoding=encoding)\n        saved_paths.append(path)\n\n    return saved_paths\n\n\ndef build_preprocessing_report(\n    raw_data: dict[str, pd.DataFrame],\n    processed_data: dict[str, pd.DataFrame],\n    relationship_checks: pd.DataFrame,\n) -> str:\n    """전처리 결과 요약 Markdown 문자열을 생성합니다."""\n    comparison = compare_shapes(raw_data, processed_data)\n    duplicate_checks = duplicate_summary(\n        processed_data,\n        key_columns={\n            "customers": "customer_id",\n            "products": "product_id",\n            "orders": "order_id",\n            "order_items": "order_item_id",\n        },\n    )\n\n    return f"""# Chapter 5 데이터 전처리 요약\n\n## 전처리 결과 파일\n\n- customers_clean.csv\n- products_clean.csv\n- orders_clean.csv\n- order_items_clean.csv\n\n## 전처리 전후 데이터 크기\n\n```text\n{comparison.to_string(index=False)}\n```\n\n## 중복 점검 결과\n\n```text\n{duplicate_checks.to_string(index=False)}\n```\n\n## 파일 간 관계 점검 결과\n\n```text\n{relationship_checks.to_string(index=False)}\n```\n\n## 주요 처리 내용\n\n- 원본 데이터는 직접 수정하지 않고 복사본을 사용했습니다.\n- 문자열 컬럼의 앞뒤 공백을 제거했습니다.\n- 고객 나이 결측치는 중앙값으로 대체했습니다.\n- 고객 도시 결측치는 Unknown으로 처리했습니다.\n- 주문 상태값 표기를 completed, cancelled, refunded 중심으로 통일했습니다.\n- 날짜 컬럼을 날짜형으로 변환하고 주문 월/요일 파생 컬럼을 만들었습니다.\n- 가격, 수량, 단가를 숫자형으로 변환했습니다.\n- 0 이하 가격, 수량, 단가는 정상 분석 대상에서 제외했습니다.\n- 주문 상세 금액 line_total 파생 컬럼을 생성했습니다.\n- 전처리 후 파일 간 키 관계를 다시 확인했습니다.\n\n## 주의 사항\n\n이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.\n"""\n', 'src/classification.py': '"""Chapter 10 leakage-aware classification analysis utilities.\n\nThe workflow protects target definition, merge integrity, model selection,\nthreshold selection, final test independence, and public-result privacy.\n\nCore contract:\n1. completed=0 and cancelled=1 only; other statuses are excluded,\n2. validate line_total = quantity * unit_price before aggregation,\n3. fail on broken order/customer joins instead of silently filling unmatched rows,\n4. keep target, identifiers, and post-outcome fields out of features,\n5. learn preprocessing inside sklearn Pipelines,\n6. select the model on validation data,\n7. select the probability threshold on validation data,\n8. freeze model and threshold before final test evaluation,\n9. separate identifier-bearing internal outputs from public predictions.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    accuracy_score,\n    classification_report,\n    confusion_matrix,\n    f1_score,\n    precision_score,\n    recall_score,\n)\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\n\n\nTARGET_COLUMN = "is_cancelled"\nALLOWED_TARGET_STATUSES = {"completed", "cancelled"}\n\nCANDIDATE_NUMERIC_FEATURES = [\n    "age",\n    "item_count",\n    "total_quantity",\n    "order_amount",\n    "order_month",\n    "order_dayofweek",\n    "days_since_signup",\n]\nCANDIDATE_CATEGORICAL_FEATURES = [\n    "gender",\n    "city",\n    "payment_method",\n]\n\nFORBIDDEN_FEATURES = {\n    "order_status",\n    TARGET_COLUMN,\n    "order_id",\n    "customer_id",\n    "product_id",\n    "cancel_reason",\n    "cancelled_at",\n}\nFORBIDDEN_REASONS = {\n    "order_status": "예측 결과와 직접 연결되는 주문 상태",\n    TARGET_COLUMN: "예측 대상 자체",\n    "order_id": "주문 식별자",\n    "customer_id": "고객 식별자",\n    "product_id": "주문 상세 식별 정보",\n    "cancel_reason": "취소 이후 생성되는 사후 정보",\n    "cancelled_at": "취소 이후 생성되는 사후 정보",\n}\nLEAKAGE_COLUMNS = {"order_status", TARGET_COLUMN}\n\nREQUIRED_COLUMNS = {\n    "customers": {"customer_id", "gender", "age", "city", "signup_date"},\n    "orders": {\n        "order_id",\n        "customer_id",\n        "order_date",\n        "order_status",\n        "payment_method",\n    },\n    "order_items": {\n        "order_id",\n        "product_id",\n        "quantity",\n        "unit_price",\n    },\n}\n\n\ndef make_one_hot_encoder() -> OneHotEncoder:\n    """Return a dense encoder compatible with multiple sklearn versions."""\n    try:\n        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)\n    except TypeError:\n        return OneHotEncoder(handle_unknown="ignore", sparse=False)\n\n\ndef load_classification_source_data(\n    processed_dir: str | Path = "data/processed",\n) -> dict[str, pd.DataFrame]:\n    """Load Chapter10 validated processed inputs without falling back to raw data."""\n    input_dir = Path(processed_dir)\n    file_map = {\n        "customers": input_dir / "customers_clean.csv",\n        "orders": input_dir / "orders_clean.csv",\n        "order_items": input_dir / "order_items_clean.csv",\n    }\n    missing = [path for path in file_map.values() if not path.exists()]\n    if missing:\n        raise FileNotFoundError(\n            "Chapter10 모델링 입력이 없습니다. 먼저 "\n            "`python scripts/prepare_ch10_data.py`를 실행하세요. "\n            + "누락 파일: "\n            + ", ".join(str(path) for path in missing)\n        )\n    return {name: pd.read_csv(path) for name, path in file_map.items()}\n\n\ndef validate_required_columns(datasets: dict[str, pd.DataFrame]) -> None:\n    """Fail fast when required datasets or columns are missing."""\n    missing_datasets = sorted(set(REQUIRED_COLUMNS) - set(datasets))\n    if missing_datasets:\n        raise KeyError(f"필수 데이터셋이 없습니다: {missing_datasets}")\n\n    for name, required in REQUIRED_COLUMNS.items():\n        missing = sorted(required - set(datasets[name].columns))\n        if missing:\n            raise KeyError(f"{name}에 필요한 컬럼이 없습니다: {missing}")\n\n\ndef validate_feature_columns(feature_columns: Iterable[str]) -> None:\n    """Reject target, identifier, and post-outcome fields from model input."""\n    columns = list(feature_columns)\n    forbidden = sorted(set(columns) & FORBIDDEN_FEATURES)\n    if forbidden:\n        raise ValueError(f"입력값에 누수/식별 위험 컬럼이 있습니다: {forbidden}")\n    duplicates = pd.Index(columns)[pd.Index(columns).duplicated()].tolist()\n    if duplicates:\n        raise ValueError(f"입력값 목록에 중복 컬럼이 있습니다: {duplicates}")\n\n\ndef _require_unique_key(df: pd.DataFrame, key: str, dataset: str) -> None:\n    missing_count = int(df[key].isna().sum())\n    duplicate_count = int(df[key].duplicated().sum())\n    if missing_count or duplicate_count:\n        raise ValueError(\n            f"{dataset}.{key} 검증 실패: "\n            f"missing={missing_count}, duplicate={duplicate_count}"\n        )\n\n\ndef build_order_item_features(order_items: pd.DataFrame) -> pd.DataFrame:\n    """Validate line totals and aggregate order items to one row per order."""\n    required = {"order_id", "product_id", "quantity", "unit_price"}\n    missing = sorted(required - set(order_items.columns))\n    if missing:\n        raise KeyError(f"order_items에 필요한 컬럼이 없습니다: {missing}")\n\n    items = order_items.copy()\n    invalid_key_or_value = items[\n        ["order_id", "product_id", "quantity", "unit_price"]\n    ].isna().any(axis=1)\n    if invalid_key_or_value.any():\n        raise ValueError(\n            "주문 특징을 만들 수 없는 주문 상세 행이 있습니다: "\n            f"{int(invalid_key_or_value.sum())}건"\n        )\n\n    items["quantity"] = pd.to_numeric(items["quantity"], errors="coerce")\n    items["unit_price"] = pd.to_numeric(items["unit_price"], errors="coerce")\n    if items[["quantity", "unit_price"]].isna().any(axis=None):\n        raise ValueError("quantity 또는 unit_price에 숫자 변환 실패가 있습니다.")\n    if (items["quantity"] <= 0).any() or (items["unit_price"] <= 0).any():\n        raise ValueError("quantity와 unit_price는 0보다 커야 합니다.")\n\n    expected_line_total = items["quantity"] * items["unit_price"]\n    if "line_total" in items.columns:\n        items["line_total"] = pd.to_numeric(items["line_total"], errors="coerce")\n        if items["line_total"].isna().any():\n            raise ValueError("order_items.line_total에 숫자 변환 실패가 있습니다.")\n        mismatch = (items["line_total"] - expected_line_total).abs().gt(1e-6)\n        if mismatch.any():\n            raise ValueError(\n                "line_total과 quantity × unit_price가 일치하지 않는 행이 있습니다: "\n                f"{int(mismatch.sum())}건"\n            )\n    else:\n        items["line_total"] = expected_line_total\n\n    if (items["line_total"] <= 0).any():\n        raise ValueError("line_total에 0 이하 값이 있습니다.")\n\n    features = (\n        items.groupby("order_id", as_index=False)\n        .agg(\n            item_count=("product_id", "size"),\n            total_quantity=("quantity", "sum"),\n            order_amount=("line_total", "sum"),\n        )\n        .sort_values("order_id")\n        .reset_index(drop=True)\n    )\n    if features["order_id"].duplicated().any():\n        raise ValueError("주문 단위 특징이 order_id당 한 행이 아닙니다.")\n    return features\n\n\ndef _checked_left_merge(\n    left: pd.DataFrame,\n    right: pd.DataFrame,\n    *,\n    on: str,\n    validate: str,\n    right_label: str,\n) -> tuple[pd.DataFrame, dict[str, object]]:\n    """Run a validated left merge and fail on row loss/growth or unmatched rows."""\n    before_rows = len(left)\n    merged = left.merge(\n        right,\n        on=on,\n        how="left",\n        validate=validate,\n        indicator=True,\n    )\n    after_rows = len(merged)\n    unmatched_count = int(merged["_merge"].eq("left_only").sum())\n    row_count_preserved = before_rows == after_rows\n\n    check = {\n        "merge": f"{on} → {right_label}",\n        "before_rows": before_rows,\n        "after_rows": after_rows,\n        "row_count_preserved": row_count_preserved,\n        "unmatched_count": unmatched_count,\n        "status": (\n            "PASS"\n            if row_count_preserved and unmatched_count == 0\n            else "FAIL"\n        ),\n    }\n\n    if not row_count_preserved or unmatched_count:\n        raise ValueError(\n            f"{on} → {right_label} 병합 검증 실패: "\n            f"before={before_rows}, after={after_rows}, "\n            f"unmatched={unmatched_count}"\n        )\n\n    return merged.drop(columns="_merge"), check\n\n\ndef build_classification_dataset(\n    customers: pd.DataFrame,\n    orders: pd.DataFrame,\n    order_items: pd.DataFrame,\n) -> tuple[\n    pd.DataFrame,\n    list[str],\n    list[str],\n    pd.DataFrame,\n    pd.DataFrame,\n]:\n    """Build strict one-row-per-order binary-classification data."""\n    datasets = {\n        "customers": customers.copy(),\n        "orders": orders.copy(),\n        "order_items": order_items.copy(),\n    }\n    validate_required_columns(datasets)\n\n    customers_data = datasets["customers"]\n    orders_data = datasets["orders"]\n    items_data = datasets["order_items"]\n\n    _require_unique_key(customers_data, "customer_id", "customers")\n    _require_unique_key(orders_data, "order_id", "orders")\n\n    required_order_values = ["order_id", "customer_id", "order_date", "order_status"]\n    missing_order_values = orders_data[required_order_values].isna().any(axis=1)\n    if missing_order_values.any():\n        raise ValueError(\n            "orders 필수값에 결측치가 있습니다: "\n            f"{int(missing_order_values.sum())}건"\n        )\n\n    orders_data["order_date"] = pd.to_datetime(\n        orders_data["order_date"], errors="coerce"\n    )\n    if orders_data["order_date"].isna().any():\n        raise ValueError(\n            "order_date 날짜 변환 실패가 있습니다: "\n            f"{int(orders_data[\'order_date\'].isna().sum())}건"\n        )\n\n    status_scope = (\n        orders_data["order_status"]\n        .value_counts(dropna=False)\n        .rename_axis("order_status")\n        .reset_index(name="order_count")\n    )\n    status_scope["used_for_binary_target"] = status_scope[\n        "order_status"\n    ].isin(ALLOWED_TARGET_STATUSES)\n\n    binary_orders = orders_data[\n        orders_data["order_status"].isin(ALLOWED_TARGET_STATUSES)\n    ].copy()\n    if binary_orders.empty:\n        raise ValueError(\n            "completed 또는 cancelled 주문이 없어 분류 데이터를 만들 수 없습니다."\n        )\n\n    binary_orders[TARGET_COLUMN] = (\n        binary_orders["order_status"].eq("cancelled").astype(int)\n    )\n    target_counts = binary_orders[TARGET_COLUMN].value_counts()\n    if set(target_counts.index) != {0, 1}:\n        raise ValueError(\n            "분류 학습에는 completed와 cancelled 주문이 모두 필요합니다."\n        )\n    if int(target_counts.min()) < 5:\n        raise ValueError(\n            "각 클래스에 최소 5개 이상의 주문이 필요합니다. "\n            f"현재 클래스별 건수: {target_counts.to_dict()}"\n        )\n\n    order_item_features = build_order_item_features(items_data)\n    model_data, order_merge_check = _checked_left_merge(\n        binary_orders,\n        order_item_features,\n        on="order_id",\n        validate="one_to_one",\n        right_label="order_item_features",\n    )\n\n    customer_lookup = customers_data[\n        ["customer_id", "gender", "age", "city", "signup_date"]\n    ].copy()\n    customer_lookup["signup_date"] = pd.to_datetime(\n        customer_lookup["signup_date"], errors="coerce"\n    )\n    if customer_lookup["signup_date"].isna().any():\n        raise ValueError(\n            "customers.signup_date 날짜 변환 실패가 있습니다: "\n            f"{int(customer_lookup[\'signup_date\'].isna().sum())}건"\n        )\n\n    model_data, customer_merge_check = _checked_left_merge(\n        model_data,\n        customer_lookup,\n        on="customer_id",\n        validate="many_to_one",\n        right_label="customers",\n    )\n\n    model_data["age"] = pd.to_numeric(model_data["age"], errors="coerce")\n    model_data["order_month"] = model_data["order_date"].dt.month\n    model_data["order_dayofweek"] = model_data["order_date"].dt.dayofweek\n    model_data["days_since_signup"] = (\n        model_data["order_date"] - model_data["signup_date"]\n    ).dt.days\n\n    negative_days = model_data["days_since_signup"].lt(0)\n    if negative_days.any():\n        raise ValueError(\n            "signup_date가 order_date보다 늦은 주문이 있습니다: "\n            f"{int(negative_days.sum())}건"\n        )\n\n    numeric_features = [\n        column\n        for column in CANDIDATE_NUMERIC_FEATURES\n        if column in model_data.columns and model_data[column].notna().any()\n    ]\n    categorical_features = [\n        column\n        for column in CANDIDATE_CATEGORICAL_FEATURES\n        if column in model_data.columns and model_data[column].notna().any()\n    ]\n    features = numeric_features + categorical_features\n    if not features:\n        raise ValueError("사용 가능한 입력 feature가 없습니다.")\n    validate_feature_columns(features)\n\n    merge_checks = pd.DataFrame([order_merge_check, customer_merge_check])\n    excluded_rows = int(\n        status_scope.loc[\n            ~status_scope["used_for_binary_target"], "order_count"\n        ].sum()\n    )\n    data_quality_checks = pd.DataFrame(\n        {\n            "check_item": [\n                "binary_target_rows",\n                "completed_rows",\n                "cancelled_rows",\n                "excluded_status_rows",\n                "line_total_mismatch",\n                "negative_days_since_signup",\n                "missing_order_item_features",\n                "missing_customer_match",\n            ],\n            "count": [\n                len(model_data),\n                int(model_data[TARGET_COLUMN].eq(0).sum()),\n                int(model_data[TARGET_COLUMN].eq(1).sum()),\n                excluded_rows,\n                0,\n                0,\n                order_merge_check["unmatched_count"],\n                customer_merge_check["unmatched_count"],\n            ],\n        }\n    )\n    status_checks = status_scope.assign(\n        check_item=lambda frame: "status_scope:" + frame["order_status"].astype(str),\n        count=lambda frame: frame["order_count"],\n    )[["check_item", "count"]]\n\n    return (\n        model_data.sort_values(["order_date", "order_id"]).reset_index(drop=True),\n        numeric_features,\n        categorical_features,\n        merge_checks,\n        pd.concat([data_quality_checks, status_checks], ignore_index=True),\n    )\n\n\ndef build_feature_audit(\n    numeric_features: list[str],\n    categorical_features: list[str],\n) -> pd.DataFrame:\n    """Document selected features and explicitly forbidden fields."""\n    selected = numeric_features + categorical_features\n    validate_feature_columns(selected)\n\n    rows = [\n        {\n            "column": column,\n            "selected": True,\n            "role": "allowed_feature",\n            "reason": "교육용 예측 시점에 사용 가능하다고 가정",\n        }\n        for column in selected\n    ]\n    rows.extend(\n        {\n            "column": column,\n            "selected": False,\n            "role": "forbidden",\n            "reason": FORBIDDEN_REASONS[column],\n        }\n        for column in sorted(FORBIDDEN_FEATURES)\n    )\n    return pd.DataFrame(rows)\n\n\ndef target_distribution(model_data: pd.DataFrame) -> pd.DataFrame:\n    """Return target class counts and ratios."""\n    counts = model_data[TARGET_COLUMN].value_counts(dropna=False).sort_index()\n    ratios = (\n        model_data[TARGET_COLUMN]\n        .value_counts(normalize=True, dropna=False)\n        .sort_index()\n    )\n    labels = {0: "completed", 1: "cancelled"}\n    return pd.DataFrame(\n        {\n            TARGET_COLUMN: counts.index,\n            "class_label": [labels.get(value, str(value)) for value in counts.index],\n            "count": counts.values,\n            "ratio": ratios.round(4).values,\n        }\n    )\n\n\ndef split_train_validation_test(\n    model_data: pd.DataFrame,\n    numeric_features: list[str],\n    categorical_features: list[str],\n    *,\n    test_size: float = 0.2,\n    validation_size: float = 0.2,\n    random_state: int = 42,\n) -> tuple[\n    pd.DataFrame,\n    pd.DataFrame,\n    pd.DataFrame,\n    pd.Series,\n    pd.Series,\n    pd.Series,\n    list[str],\n]:\n    """Create educational stratified train/validation/test splits."""\n    if test_size <= 0 or validation_size <= 0:\n        raise ValueError("test_size와 validation_size는 0보다 커야 합니다.")\n    if test_size + validation_size >= 1:\n        raise ValueError("test_size와 validation_size의 합은 1보다 작아야 합니다.")\n\n    features = numeric_features + categorical_features\n    validate_feature_columns(features)\n    X = model_data[features].copy()\n    y = model_data[TARGET_COLUMN].copy()\n\n    if set(y.dropna().unique()) != {0, 1}:\n        raise ValueError("이진 분류에는 0과 1 두 타깃 클래스가 필요합니다.")\n\n    X_train_valid, X_test, y_train_valid, y_test = train_test_split(\n        X,\n        y,\n        test_size=test_size,\n        random_state=random_state,\n        stratify=y,\n    )\n    validation_ratio = validation_size / (1 - test_size)\n    X_train, X_valid, y_train, y_valid = train_test_split(\n        X_train_valid,\n        y_train_valid,\n        test_size=validation_ratio,\n        random_state=random_state,\n        stratify=y_train_valid,\n    )\n\n    for split_name, target in {\n        "train": y_train,\n        "validation": y_valid,\n        "test": y_test,\n    }.items():\n        if set(target.unique()) != {0, 1}:\n            raise ValueError(\n                f"{split_name} 데이터에 두 클래스가 모두 포함되지 않았습니다."\n            )\n\n    return (\n        X_train,\n        X_valid,\n        X_test,\n        y_train,\n        y_valid,\n        y_test,\n        features,\n    )\n\n\ndef build_split_summary(\n    y_train: pd.Series,\n    y_valid: pd.Series,\n    y_test: pd.Series,\n) -> pd.DataFrame:\n    """Summarize class counts and ratios for each split."""\n    rows: list[dict[str, object]] = []\n    for split_name, target in {\n        "train": y_train,\n        "validation": y_valid,\n        "test": y_test,\n    }.items():\n        counts = target.value_counts().sort_index()\n        total = len(target)\n        for class_value in [0, 1]:\n            count = int(counts.get(class_value, 0))\n            rows.append(\n                {\n                    "split": split_name,\n                    TARGET_COLUMN: class_value,\n                    "class_label": (\n                        "cancelled" if class_value == 1 else "completed"\n                    ),\n                    "count": count,\n                    "ratio": round(count / total, 4) if total else 0,\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef make_preprocessor(\n    numeric_features: list[str],\n    categorical_features: list[str],\n) -> ColumnTransformer:\n    """Create train-only numeric and categorical preprocessing."""\n    transformers: list[tuple[str, Pipeline, list[str]]] = []\n\n    if numeric_features:\n        numeric = Pipeline(\n            steps=[\n                ("imputer", SimpleImputer(strategy="median")),\n                ("scaler", StandardScaler()),\n            ]\n        )\n        transformers.append(("num", numeric, numeric_features))\n\n    if categorical_features:\n        categorical = Pipeline(\n            steps=[\n                ("imputer", SimpleImputer(strategy="most_frequent")),\n                ("onehot", make_one_hot_encoder()),\n            ]\n        )\n        transformers.append(("cat", categorical, categorical_features))\n\n    if not transformers:\n        raise ValueError("전처리할 feature가 없습니다.")\n    return ColumnTransformer(transformers=transformers)\n\n\ndef make_classification_models(\n    numeric_features: list[str],\n    categorical_features: list[str],\n    random_state: int = 42,\n) -> dict[str, Pipeline]:\n    """Create the fixed baseline and candidate models."""\n    return {\n        "Dummy Most Frequent": Pipeline(\n            steps=[\n                (\n                    "preprocessor",\n                    make_preprocessor(numeric_features, categorical_features),\n                ),\n                ("model", DummyClassifier(strategy="most_frequent")),\n            ]\n        ),\n        "Logistic Regression": Pipeline(\n            steps=[\n                (\n                    "preprocessor",\n                    make_preprocessor(numeric_features, categorical_features),\n                ),\n                (\n                    "model",\n                    LogisticRegression(max_iter=1000, class_weight="balanced"),\n                ),\n            ]\n        ),\n        "Random Forest": Pipeline(\n            steps=[\n                (\n                    "preprocessor",\n                    make_preprocessor(numeric_features, categorical_features),\n                ),\n                (\n                    "model",\n                    RandomForestClassifier(\n                        n_estimators=200,\n                        random_state=random_state,\n                        class_weight="balanced",\n                    ),\n                ),\n            ]\n        ),\n    }\n\n\ndef evaluate_classification_predictions(\n    y_true: pd.Series,\n    y_pred: np.ndarray,\n) -> dict[str, float]:\n    """Evaluate predictions with accuracy, precision, recall, and F1."""\n    return {\n        "accuracy": float(accuracy_score(y_true, y_pred)),\n        "precision": float(precision_score(y_true, y_pred, zero_division=0)),\n        "recall": float(recall_score(y_true, y_pred, zero_division=0)),\n        "f1": float(f1_score(y_true, y_pred, zero_division=0)),\n    }\n\n\ndef train_and_compare_on_validation(\n    X_train: pd.DataFrame,\n    X_valid: pd.DataFrame,\n    y_train: pd.Series,\n    y_valid: pd.Series,\n    numeric_features: list[str],\n    categorical_features: list[str],\n    random_state: int = 42,\n) -> tuple[\n    dict[str, Pipeline],\n    pd.DataFrame,\n    dict[str, np.ndarray],\n    dict[str, np.ndarray],\n]:\n    """Fit on train and compare fixed candidates on validation only."""\n    validate_feature_columns(X_train.columns)\n    if list(X_train.columns) != list(X_valid.columns):\n        raise ValueError("Train과 Validation 입력 컬럼 구성이 다릅니다.")\n\n    models = make_classification_models(\n        numeric_features=numeric_features,\n        categorical_features=categorical_features,\n        random_state=random_state,\n    )\n    rows: list[dict[str, Any]] = []\n    predictions: dict[str, np.ndarray] = {}\n    probabilities: dict[str, np.ndarray] = {}\n\n    for model_name, model in models.items():\n        model.fit(X_train, y_train)\n        y_pred = model.predict(X_valid)\n        rows.append(\n            {\n                "model": model_name,\n                "evaluation_split": "validation",\n                **evaluate_classification_predictions(y_valid, y_pred),\n            }\n        )\n        predictions[model_name] = y_pred\n        if hasattr(model, "predict_proba"):\n            probabilities[model_name] = model.predict_proba(X_valid)[:, 1]\n\n    comparison = (\n        pd.DataFrame(rows)\n        .sort_values(\n            ["f1", "recall", "precision", "model"],\n            ascending=[False, False, False, True],\n        )\n        .reset_index(drop=True)\n    )\n    return models, comparison, predictions, probabilities\n\n\ndef select_validation_model(validation_comparison: pd.DataFrame) -> str:\n    """Freeze the best non-dummy model using validation metrics only."""\n    required = {"model", "f1", "recall", "precision"}\n    missing = sorted(required - set(validation_comparison.columns))\n    if missing:\n        raise KeyError(f"Validation 모델 비교표에 필요한 컬럼이 없습니다: {missing}")\n\n    candidates = validation_comparison.loc[\n        ~validation_comparison["model"].eq("Dummy Most Frequent")\n    ].sort_values(\n        ["f1", "recall", "precision", "model"],\n        ascending=[False, False, False, True],\n    )\n    if candidates.empty:\n        raise ValueError("선택할 비베이스라인 모델이 없습니다.")\n    return str(candidates.iloc[0]["model"])\n\n\ndef threshold_metrics(\n    y_true: pd.Series,\n    y_proba: np.ndarray,\n    thresholds: list[float] | None = None,\n) -> pd.DataFrame:\n    """Evaluate probability thresholds on validation data."""\n    thresholds = thresholds or [\n        round(value, 2) for value in np.arange(0.2, 0.81, 0.05)\n    ]\n    rows = []\n    for threshold in thresholds:\n        y_pred = (y_proba >= threshold).astype(int)\n        rows.append(\n            {\n                "threshold": float(threshold),\n                **evaluate_classification_predictions(y_true, y_pred),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef choose_threshold(\n    threshold_df: pd.DataFrame,\n    *,\n    default_threshold: float = 0.5,\n) -> float:\n    """Choose validation threshold by F1, recall, precision, then lower threshold."""\n    if threshold_df.empty:\n        return default_threshold\n    ranked = threshold_df.sort_values(\n        ["f1", "recall", "precision", "threshold"],\n        ascending=[False, False, False, True],\n    )\n    return float(ranked.iloc[0]["threshold"])\n\n\ndef build_selection_summary(\n    selected_model_name: str,\n    selected_threshold: float,\n    validation_comparison: pd.DataFrame,\n    threshold_df: pd.DataFrame,\n) -> pd.DataFrame:\n    """Record that model and threshold selection occurred on validation data."""\n    model_row = validation_comparison.loc[\n        validation_comparison["model"].eq(selected_model_name)\n    ]\n    threshold_match = np.isclose(\n        threshold_df["threshold"].astype(float).to_numpy(),\n        float(selected_threshold),\n    )\n    if model_row.empty or not threshold_match.any():\n        raise ValueError("선택 모델 또는 임계값이 validation evidence에 없습니다.")\n    return pd.DataFrame(\n        [\n            {\n                "selection_item": "model",\n                "value": selected_model_name,\n                "selected_on": "validation",\n                "criterion": "f1 → recall → precision",\n            },\n            {\n                "selection_item": "threshold",\n                "value": selected_threshold,\n                "selected_on": "validation",\n                "criterion": "f1 → recall → precision → lower threshold",\n            },\n        ]\n    )\n\n\ndef final_test_evaluation(\n    model: Pipeline,\n    X_test: pd.DataFrame,\n    y_test: pd.Series,\n    *,\n    threshold: float,\n    model_name: str | None = None,\n) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:\n    """Evaluate one frozen model/threshold pair on final test.\n\n    Chapter 10 legacy callers omit ``model_name`` and keep the historical\n    ``evaluation_split=\'test\'`` label. Chapter 15 supplies ``model_name`` and\n    receives explicit frozen-before-test evidence.\n    """\n    if not 0 <= threshold <= 1:\n        raise ValueError("threshold는 0과 1 사이여야 합니다.")\n    y_proba = model.predict_proba(X_test)[:, 1]\n    y_pred = (y_proba >= threshold).astype(int)\n    row: dict[str, Any] = {\n        "evaluation_split": "test_final" if model_name is not None else "test",\n        "threshold": threshold,\n        "selection_status": "frozen_before_test",\n        **evaluate_classification_predictions(y_test, y_pred),\n    }\n    if model_name is not None:\n        row = {"model": model_name, **row}\n    return y_pred, y_proba, pd.DataFrame([row])\n\n\ndef confusion_matrix_dataframe(\n    y_true: pd.Series,\n    y_pred: np.ndarray,\n) -> pd.DataFrame:\n    """Return a readable binary confusion matrix."""\n    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])\n    return pd.DataFrame(\n        matrix,\n        index=["actual_completed", "actual_cancelled"],\n        columns=["pred_completed", "pred_cancelled"],\n    )\n\n\ndef classification_report_dataframe(\n    y_true: pd.Series,\n    y_pred: np.ndarray,\n) -> pd.DataFrame:\n    """Return sklearn classification_report as a DataFrame."""\n    report = classification_report(\n        y_true,\n        y_pred,\n        labels=[0, 1],\n        target_names=["completed", "cancelled"],\n        zero_division=0,\n        output_dict=True,\n    )\n    return pd.DataFrame(report).T.reset_index().rename(columns={"index": "label"})\n\n\ndef create_prediction_result(\n    y_test: pd.Series,\n    y_pred: np.ndarray,\n    y_proba: np.ndarray,\n    *,\n    model_name: str,\n    threshold: float,\n    source_index: pd.Index | None = None,\n) -> pd.DataFrame:\n    """Create prediction evidence with optional internal source traceability.\n\n    Chapter 10 can pass ``source_index`` for an internal-only artifact. Chapter\n    15 omits it, so the returned table is public-safe by construction.\n    """\n    if len(y_test) != len(y_pred) or len(y_pred) != len(y_proba):\n        raise ValueError("테스트 실제값·예측값·확률의 길이가 일치하지 않습니다.")\n    if source_index is not None and len(source_index) != len(y_test):\n        raise ValueError("source_index와 테스트 결과 길이가 일치하지 않습니다.")\n\n    result = pd.DataFrame(\n        {\n            "record_id": [\n                f"test_{position:04d}" for position in range(1, len(y_test) + 1)\n            ],\n            "actual_is_cancelled": y_test.to_numpy(),\n            "predicted_is_cancelled": y_pred,\n            "cancel_probability": y_proba,\n            "model": model_name,\n            "threshold": threshold,\n        }\n    )\n    if source_index is not None:\n        result.insert(1, "source_index", source_index.to_numpy())\n    return result\n\n\ndef public_prediction_result(prediction_result: pd.DataFrame) -> pd.DataFrame:\n    """Remove internal source references from public predictions."""\n    public_columns = [\n        "record_id",\n        "actual_is_cancelled",\n        "predicted_is_cancelled",\n        "cancel_probability",\n        "model",\n        "threshold",\n    ]\n    return prediction_result[public_columns].copy()\n\n\ndef build_classification_checklist() -> pd.DataFrame:\n    """Return the human-review checklist for classification and LLM code."""\n    items = [\n        "completed와 cancelled만 사용해 이진 타깃을 만들었는가?",\n        "refunded 등 다른 상태를 0 클래스에 섞지 않았는가?",\n        "예측 시점과 feature 가용성 가정을 설명할 수 있는가?",\n        "line_total = quantity × unit_price를 검증했는가?",\n        "병합에 validate를 사용하고 미매칭 0건을 확인했는가?",\n        "order_status, target, ID, 사후 정보를 feature에서 제외했는가?",\n        "train, validation, test를 분리했는가?",\n        "모델과 threshold는 validation에서 선택했는가?",\n        "모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?",\n        "Dummy 기준 모델과 비교했는가?",\n        "accuracy 외 precision, recall, F1과 FP/FN을 함께 확인했는가?",\n        "공개 prediction에서 내부 source index와 식별자를 제거했는가?",\n        "random split의 교육용 한계를 기록했는가?",\n        "모델 결과를 취소 원인으로 단정하지 않았는가?",\n    ]\n    return pd.DataFrame({"check_item": items, "status": ["□"] * len(items)})\n\n\ndef build_classification_validation(\n    *,\n    model_data: pd.DataFrame,\n    features: list[str],\n    merge_checks: pd.DataFrame,\n    y_train: pd.Series,\n    y_valid: pd.Series,\n    y_test: pd.Series,\n    validation_comparison: pd.DataFrame,\n    selected_model_name: str,\n    threshold_df: pd.DataFrame,\n    selected_threshold: float,\n    test_metrics: pd.DataFrame | None = None,\n    prediction_result: pd.DataFrame | None = None,\n    prediction_result_public: pd.DataFrame | None = None,\n) -> pd.DataFrame:\n    """Create machine-checkable evidence for Chapter 10 and Chapter 15.\n\n    ``prediction_result_public`` is retained for the Chapter 10 pipeline.\n    Chapter 15 passes ``prediction_result`` and ``test_metrics`` so the final\n    test frozen-decision contract is also validated.\n    """\n    target_values = set(model_data[TARGET_COLUMN].dropna().unique())\n    feature_forbidden = sorted(set(features) & set(FORBIDDEN_FEATURES))\n    merge_pass = bool(\n        not merge_checks.empty\n        and merge_checks["row_count_preserved"].eq(True).all()\n        and merge_checks["unmatched_count"].eq(0).all()\n    )\n    split_has_both = all(\n        set(target.unique()) == {0, 1}\n        for target in [y_train, y_valid, y_test]\n    )\n    selected_in_validation = bool(\n        selected_model_name in set(validation_comparison["model"])\n        and selected_model_name != "Dummy Most Frequent"\n    )\n    threshold_in_validation = bool(\n        np.isclose(\n            threshold_df["threshold"].astype(float).to_numpy(),\n            float(selected_threshold),\n        ).any()\n    )\n\n    public_frame = prediction_result if prediction_result is not None else prediction_result_public\n    public_forbidden_columns = {\n        "source_index",\n        "order_id",\n        "customer_id",\n        "product_id",\n        "name",\n        "email",\n        "phone",\n        "address",\n    }\n    public_leak = (\n        sorted(public_forbidden_columns & set(public_frame.columns))\n        if public_frame is not None\n        else ["prediction_result_missing"]\n    )\n\n    rows: list[dict[str, object]] = [\n        {\n            "check": "target_exactly_completed_cancelled",\n            "value": sorted(target_values),\n            "status": "PASS" if target_values == {0, 1} else "FAIL",\n        },\n        {\n            "check": "forbidden_feature_overlap",\n            "value": len(feature_forbidden),\n            "status": "PASS" if not feature_forbidden else "FAIL",\n        },\n        {\n            "check": "merge_rows_and_matches",\n            "value": merge_pass,\n            "status": "PASS" if merge_pass else "FAIL",\n        },\n        {\n            "check": "all_splits_have_both_classes",\n            "value": split_has_both,\n            "status": "PASS" if split_has_both else "FAIL",\n        },\n        {\n            "check": "model_selected_on_validation",\n            "value": selected_in_validation,\n            "status": "PASS" if selected_in_validation else "FAIL",\n        },\n        {\n            "check": "threshold_selected_on_validation",\n            "value": threshold_in_validation,\n            "status": "PASS" if threshold_in_validation else "FAIL",\n        },\n        {\n            "check": "public_prediction_identifier_columns",\n            "value": ",".join(public_leak) if public_leak else "none",\n            "status": "PASS" if not public_leak else "FAIL",\n        },\n    ]\n\n    if test_metrics is not None:\n        test_is_final = bool(\n            len(test_metrics) == 1\n            and "evaluation_split" in test_metrics.columns\n            and test_metrics["evaluation_split"].eq("test_final").all()\n            and "selection_status" in test_metrics.columns\n            and test_metrics["selection_status"].eq("frozen_before_test").all()\n        )\n        rows.append(\n            {\n                "check": "test_used_as_final_evaluation",\n                "value": test_is_final,\n                "status": "PASS" if test_is_final else "FAIL",\n            }\n        )\n\n    validation = pd.DataFrame(rows)\n    failed = validation.loc[validation["status"].eq("FAIL")]\n    if not failed.empty:\n        raise ValueError(\n            "분류 분석 핵심 검증에 실패했습니다:\\n"\n            + failed.to_string(index=False)\n        )\n    return validation\n\n\ndef build_classification_report_text(\n    model_data: pd.DataFrame,\n    target_dist: pd.DataFrame,\n    feature_audit: pd.DataFrame,\n    merge_checks: pd.DataFrame,\n    split_summary: pd.DataFrame,\n    validation_comparison: pd.DataFrame,\n    selected_model_name: str,\n    selected_threshold: float,\n    threshold_df: pd.DataFrame,\n    test_metrics: pd.DataFrame,\n    confusion_df: pd.DataFrame,\n    validation: pd.DataFrame,\n    checklist: pd.DataFrame,\n) -> str:\n    """Build the public Markdown summary report."""\n    return f"""# Chapter 10 분류 분석 요약 보고서\n\n## 1. 분석 목적과 타깃\n완료 주문과 취소 주문만 사용해 주문 취소 여부를 예측합니다.\n\n- completed = 0\n- cancelled = 1\n- refunded 및 기타 상태 = 모델링 범위 제외\n\n## 2. 모델링 데이터\n- 행 수: {model_data.shape[0]}\n- 예측 대상: {TARGET_COLUMN}\n\n## 3. 타깃 클래스 분포\n```text\n{target_dist.to_string(index=False)}\n```\n\n## 4. Feature Audit\n```text\n{feature_audit.to_string(index=False)}\n```\n\n## 5. 병합 검증\n```text\n{merge_checks.to_string(index=False)}\n```\n\n## 6. Train / Validation / Test\n```text\n{split_summary.to_string(index=False)}\n```\n\n## 7. Validation 모델 비교\n```text\n{validation_comparison.to_string(index=False)}\n```\n\n선택 모델: **{selected_model_name}**\n\n## 8. Validation Threshold 비교\n```text\n{threshold_df.to_string(index=False)}\n```\n\n선택 threshold: **{selected_threshold:.2f}**\n\n## 9. Final Test\n```text\n{test_metrics.to_string(index=False)}\n```\n\n## 10. Confusion Matrix\n```text\n{confusion_df.to_string()}\n```\n\n## 11. 자동 Validation Evidence\n```text\n{validation.to_string(index=False)}\n```\n\n## 12. 사람 검토 체크리스트\n```text\n{checklist.to_string(index=False)}\n```\n\n## 13. 해석 시 주의사항\n- 모델과 threshold는 Validation에서 선택했습니다.\n- Final Test는 선택이 끝난 뒤 마지막 평가에만 사용했습니다.\n- accuracy뿐 아니라 precision, recall, F1과 FP/FN을 함께 봅니다.\n- random split은 교육용 설계이며 실제 운영 전에는 out-of-time 평가가 필요합니다.\n- 모델이 학습한 예측 패턴을 취소의 원인으로 단정하지 않습니다.\n- 공개 prediction에는 내부 source index나 원본 식별자를 포함하지 않습니다.\n"""\n\n\ndef save_classification_outputs(\n    *,\n    model_data: pd.DataFrame,\n    target_dist: pd.DataFrame,\n    feature_audit: pd.DataFrame,\n    merge_checks: pd.DataFrame,\n    data_quality_checks: pd.DataFrame,\n    split_summary: pd.DataFrame,\n    validation_comparison: pd.DataFrame,\n    threshold_df: pd.DataFrame,\n    test_metrics: pd.DataFrame,\n    prediction_result_internal: pd.DataFrame,\n    prediction_result_public: pd.DataFrame,\n    confusion_df: pd.DataFrame,\n    report_df: pd.DataFrame,\n    validation: pd.DataFrame,\n    checklist: pd.DataFrame,\n    selected_model_name: str,\n    selected_threshold: float,\n    report_dir: str | Path = "reports",\n) -> dict[str, Path]:\n    """Save internal evidence and privacy-safe public outputs separately."""\n    output_dir = Path(report_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    internal_columns = [\n        column\n        for column in [\n            "order_id",\n            "customer_id",\n            TARGET_COLUMN,\n            *CANDIDATE_NUMERIC_FEATURES,\n            *CANDIDATE_CATEGORICAL_FEATURES,\n        ]\n        if column in model_data.columns\n    ]\n    internal_model_data = model_data[internal_columns].copy()\n\n    paths = {\n        "model_data_internal": (\n            output_dir / "ch10_classification_model_data_internal.csv"\n        ),\n        "target_distribution": output_dir / "ch10_target_distribution.csv",\n        "feature_audit": output_dir / "ch10_feature_audit.csv",\n        "merge_checks": output_dir / "ch10_merge_checks.csv",\n        "data_quality_checks": output_dir / "ch10_data_quality_checks.csv",\n        "split_summary": output_dir / "ch10_split_summary.csv",\n        "validation_comparison": (\n            output_dir / "ch10_validation_model_comparison.csv"\n        ),\n        "threshold_metrics": (\n            output_dir / "ch10_validation_threshold_metrics.csv"\n        ),\n        "test_metrics": output_dir / "ch10_test_metrics.csv",\n        "predictions_internal": (\n            output_dir / "ch10_classification_predictions_internal.csv"\n        ),\n        "predictions": output_dir / "ch10_classification_predictions.csv",\n        "confusion_matrix": output_dir / "ch10_confusion_matrix.csv",\n        "classification_report": (\n            output_dir / "ch10_classification_report.csv"\n        ),\n        "validation": output_dir / "ch10_classification_validation.csv",\n        "checklist": output_dir / "ch10_classification_checklist.csv",\n        "report": output_dir / "ch10_classification_summary.md",\n    }\n\n    csv_objects = {\n        "model_data_internal": internal_model_data,\n        "target_distribution": target_dist,\n        "feature_audit": feature_audit,\n        "merge_checks": merge_checks,\n        "data_quality_checks": data_quality_checks,\n        "split_summary": split_summary,\n        "validation_comparison": validation_comparison,\n        "threshold_metrics": threshold_df,\n        "test_metrics": test_metrics,\n        "predictions_internal": prediction_result_internal,\n        "predictions": prediction_result_public,\n        "classification_report": report_df,\n        "validation": validation,\n        "checklist": checklist,\n    }\n    for key, frame in csv_objects.items():\n        frame.to_csv(paths[key], index=False, encoding="utf-8-sig")\n    confusion_df.to_csv(paths["confusion_matrix"], encoding="utf-8-sig")\n\n    report_text = build_classification_report_text(\n        model_data=model_data,\n        target_dist=target_dist,\n        feature_audit=feature_audit,\n        merge_checks=merge_checks,\n        split_summary=split_summary,\n        validation_comparison=validation_comparison,\n        selected_model_name=selected_model_name,\n        selected_threshold=selected_threshold,\n        threshold_df=threshold_df,\n        test_metrics=test_metrics,\n        confusion_df=confusion_df,\n        validation=validation,\n        checklist=checklist,\n    )\n    paths["report"].write_text(report_text, encoding="utf-8")\n    return paths\n\n\ndef run_classification_analysis(processed_dir="data/processed",report_dir="reports",random_state=42):\n    from src.ch10_review import run\n    return run(processed_dir,report_dir,random_state)\n', 'scripts/prepare_ch10_data.py': '"""Prepare validated Chapter10 classification input from common project raw data.\n\nRun from the project root:\n\n    python scripts/prepare_ch10_data.py\n\nInput:\n    data/raw/customers.csv\n    data/raw/products.csv\n    data/raw/orders.csv\n    data/raw/order_items.csv\n\nOutput:\n    data/processed/customers_clean.csv\n    data/processed/products_clean.csv\n    data/processed/orders_clean.csv\n    data/processed/order_items_clean.csv\n\nChapter05의 전용 오류 탐지 데이터(`practice/chapter05/data/raw`)는 사용하지 않습니다.\nChapter10은 Chapter08·09와 같은 공통 프로젝트 데이터 흐름을 이어받습니다.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nimport sys\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.data_loader import load_sales_data  # noqa: E402\nfrom src.preprocessing import (  # noqa: E402\n    preprocess_sales_data,\n    save_processed_data,\n    validate_relationships,\n)\n\n\nRAW_DIR = PROJECT_ROOT / "data" / "raw"\nPROCESSED_DIR = PROJECT_ROOT / "data" / "processed"\n\n\ndef prepare_ch10_data(\n    raw_dir: str | Path = RAW_DIR,\n    processed_dir: str | Path = PROCESSED_DIR,\n) -> dict[str, object]:\n    """Preprocess common raw data and fail on core relationship errors."""\n    raw_data = load_sales_data(raw_dir)\n    processed_data = preprocess_sales_data(raw_data)\n    relationship_checks = validate_relationships(processed_data)\n\n    if (\n        not relationship_checks.empty\n        and not relationship_checks["invalid_count"].eq(0).all()\n    ):\n        failed = relationship_checks.loc[\n            relationship_checks["invalid_count"].ne(0)\n        ]\n        raise ValueError(\n            "Chapter10 입력 관계 검증에 실패했습니다:\\n"\n            + failed.to_string(index=False)\n        )\n\n    saved_paths = save_processed_data(processed_data, processed_dir)\n    return {\n        "raw_data": raw_data,\n        "processed_data": processed_data,\n        "relationship_checks": relationship_checks,\n        "saved_paths": saved_paths,\n    }\n\n\ndef main() -> None:\n    result = prepare_ch10_data()\n\n    print("Chapter10 모델링 입력 준비 완료")\n    print(f"입력 Raw 경로: {RAW_DIR}")\n    print("\\n[관계 검증]")\n    print(result["relationship_checks"].to_string(index=False))\n    print("\\n[저장된 processed 파일]")\n    for path in result["saved_paths"]:\n        print(f"- {path}")\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/run_classification_analysis.py': '"""Chapter 10 분류 분석 실행 스크립트.\n\n실행 방법:\n    python scripts/prepare_ch10_data.py\n    python scripts/run_classification_analysis.py\n\n입력:\n    data/processed/customers_clean.csv\n    data/processed/orders_clean.csv\n    data/processed/order_items_clean.csv\n\n주요 출력:\n    reports/ch10_classification_model_data_internal.csv\n    reports/ch10_target_distribution.csv\n    reports/ch10_feature_audit.csv\n    reports/ch10_merge_checks.csv\n    reports/ch10_data_quality_checks.csv\n    reports/ch10_split_summary.csv\n    reports/ch10_validation_model_comparison.csv\n    reports/ch10_validation_threshold_metrics.csv\n    reports/ch10_test_metrics.csv\n    reports/ch10_classification_predictions_internal.csv\n    reports/ch10_classification_predictions.csv\n    reports/ch10_confusion_matrix.csv\n    reports/ch10_classification_report.csv\n    reports/ch10_classification_validation.csv\n    reports/ch10_classification_checklist.csv\n    reports/ch10_classification_summary.md\n"""\n\nfrom pathlib import Path\nimport sys\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.classification import run_classification_analysis\n\n\nPROCESSED_DIR = Path("data/processed")\nREPORT_DIR = Path("reports")\n\n\ndef main() -> None:\n    """10장 분류 분석 전체 파이프라인을 실행합니다."""\n    result = run_classification_analysis(\n        processed_dir=PROCESSED_DIR,\n        report_dir=REPORT_DIR,\n        random_state=42,\n    )\n\n    print("Chapter10 분류 분석 완료")\n\n    print("\\n[모델링 데이터]")\n    print(result["model_data"].shape)\n\n    print("\\n[타깃 분포]")\n    print(result["target_distribution"].to_string(index=False))\n\n    print("\\n[Feature Audit]")\n    print(result["feature_audit"].to_string(index=False))\n\n    print("\\n[병합 검증]")\n    print(result["merge_checks"].to_string(index=False))\n\n    print("\\n[데이터 분할]")\n    print(result["split_summary"].to_string(index=False))\n\n    print("\\n[Validation 모델 비교]")\n    print(result["validation_model_comparison"].to_string(index=False))\n\n    print("\\n[Validation에서 고정한 모델과 Threshold]")\n    print("모델:", result["selected_model_name"])\n    print("Threshold:", result["selected_threshold"])\n\n    print("\\n[Validation Threshold 비교]")\n    print(result["threshold_metrics"].to_string(index=False))\n\n    print("\\n[Final Test 성능]")\n    print(result["test_metrics"].to_string(index=False))\n\n    print("\\n[Final Test 혼동행렬]")\n    print(result["confusion_matrix"].to_string())\n\n    print("\\n[자동 Validation Evidence]")\n    print(result["validation"].to_string(index=False))\n\n    print("\\n[공개 Prediction 컬럼]")\n    print(result["prediction_result"].columns.tolist())\n\n    print("\\n[저장된 결과 파일]")\n    for name, path in result["output_paths"].items():\n        print(f"- {name}: {path}")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/ch10_review.py': '"""Chapter10 supplemental evidence; original reference functions are preserved."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport numpy as np\nimport pandas as pd\nfrom src import classification as c\n\nPOLICY = {\n \'prediction_time\': \'order details and payment method fixed, before final cancellation outcome\',\n \'target\': \'completed=0,cancelled=1; refunded excluded\',\n \'eligibility\': \'hold binary orders before signup; no date correction; 203 eligible + 45 hold + 52 target-excluded = 300\',\n \'features\': \'7 numeric + 3 categorical; no status,target,IDs,post-outcome data\',\n \'random_state\': 42,\n \'split\': \'stratified train60/validation20/test20\',\n \'model_rule\': \'non-Dummy Validation F1 desc, recall desc, precision desc, model asc\',\n \'threshold_rule\': \'0.20:0.05:0.80; Validation F1 desc, recall desc, precision desc, threshold asc\',\n \'cost_assumption\': \'unknown FP/FN costs; educational F1 rule, not optimized business cost\',\n \'usage\': \'hold; future-time validation and data semantics unresolved\',\n}\nAUDIT = {}\nEVENTS = []\n_original_build = c.build_classification_dataset\n\ndef audited_dataset(customers, orders, order_items):\n    # Fail on key/date defects before the pre-declared eligibility hold.\n    for df,key in [(customers,\'customer_id\'),(orders,\'order_id\'),(order_items,\'order_item_id\')]:\n        assert df[key].notna().all() and not df[key].duplicated().any(), key\n    assert orders.customer_id.isin(customers.customer_id).all()\n    assert order_items.order_id.isin(orders.order_id).all()\n    dates = orders[[\'order_id\',\'customer_id\',\'order_date\',\'order_status\']].merge(\n        customers[[\'customer_id\',\'signup_date\']],on=\'customer_id\',how=\'left\',validate=\'many_to_one\')\n    for col in [\'order_date\',\'signup_date\']:\n        dates[col]=pd.to_datetime(dates[col],errors=\'coerce\')\n        assert dates[col].notna().all(), col\n    binary=dates.order_status.isin([\'completed\',\'cancelled\'])\n    before=dates.order_date.lt(dates.signup_date)\n    held=dates.loc[binary & before].copy()\n    eligible=dates.loc[binary & ~before]\n    excluded=dates.loc[~binary]\n    assert len(eligible)+len(held)+len(excluded)==len(orders)\n    assert (len(orders),len(eligible),len(held),len(excluded))==(300,203,45,52)\n    held[\'hold_reason\']=\'order_date_before_signup_date_business_meaning_unresolved\'\n    amount=order_items.groupby(\'order_id\').agg(detail_rows=(\'order_item_id\',\'size\'),quantity=(\'quantity\',\'sum\'),amount=(\'line_total\',\'sum\'))\n    rows=[]\n    for scope,frame in [(\'eligible\',eligible),(\'business_date_hold\',held),(\'target_excluded\',excluded)]:\n        for status, group in frame.groupby(\'order_status\'):\n            detail=amount.loc[group.order_id]\n            rows.append(dict(scope=scope,order_status=status,order_count=len(group),customer_count=group.customer_id.nunique(),detail_rows=int(detail.detail_rows.sum()),total_quantity=float(detail.quantity.sum()),order_amount=float(detail.amount.sum())))\n    scope=pd.DataFrame(rows)\n    kept=orders.loc[~orders.order_id.isin(held.order_id)].copy()\n    result=list(_original_build(customers,kept,order_items))\n    result[3][\'validate_relationship\']=[\'one_to_one\',\'many_to_one\']\n    result[4]=pd.concat([result[4],pd.DataFrame([\n      {\'check_item\':\'all_source_orders\',\'count\':len(orders)},\n      {\'check_item\':\'binary_before_business_hold\',\'count\':int(binary.sum())},\n      {\'check_item\':\'unresolved_business_date_hold\',\'count\':len(held)},\n      {\'check_item\':\'source_early_orders_all_statuses\',\'count\':int(before.sum())},\n      {\'check_item\':\'order_date_parse_failure\',\'count\':0},\n      {\'check_item\':\'signup_date_parse_failure\',\'count\':0},\n    ])],ignore_index=True)\n    AUDIT.update(scope=scope,held=held)\n    return tuple(result)\n\nc.build_classification_dataset=audited_dataset\n\ndef frame_hash(frame):\n    return hashlib.sha256(frame.to_csv(index=True,float_format=\'%.15g\').encode()).hexdigest()\n\ndef freeze_selection(model, threshold, comparison, threshold_df, model_data, splits, outdir, expected=None):\n    assert not EVENTS or EVENTS[-1] != \'final_test_evaluated\'\n    frozen={\'policy\':POLICY,\'model\':model,\'threshold\':float(threshold),\n      \'dataset_hash\':frame_hash(model_data),\n      \'validation_hash\':frame_hash(comparison),\'threshold_grid_hash\':frame_hash(threshold_df),\n      \'split_hashes\':{name:hashlib.sha256(\',\'.join(map(str,x.index)).encode()).hexdigest() for name,x in zip([\'train\',\'validation\',\'test\'],splits)}}\n    if expected is not None:\n        assert frozen==expected, \'Pre-Test frozen selection or data changed; stop without evaluating Test\'\n    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n    p=outdir/\'ch10_selection_frozen_internal.json\'\n    p.write_text(json.dumps(frozen,ensure_ascii=False,indent=2),encoding=\'utf8\')\n    assert json.loads(p.read_text())==frozen\n    c.build_selection_summary(model,threshold,comparison,threshold_df).to_csv(outdir/\'ch10_selection_summary.csv\',index=False)\n    EVENTS.append(\'selection_saved_before_test\')\n    return frozen\n\n_original_final=c.final_test_evaluation\n\ndef final_once(*args,**kwargs):\n    assert EVENTS and EVENTS[-1]==\'selection_saved_before_test\', \'Freeze required before Test\'\n    assert EVENTS.count(\'final_test_evaluated\')==0, \'Only one Test evaluation per process\'\n    result=_original_final(*args,**kwargs)\n    EVENTS.append(\'final_test_evaluated\')\n    return result\nc.final_test_evaluation=final_once\n\ndef fit_audit(models,X_train,numeric,categorical):\n    rows=[]\n    for name,model in models.items():\n        pre=model.named_steps[\'preprocessor\']\n        num=pre.named_transformers_[\'num\']\n        np.testing.assert_allclose(num.named_steps[\'imputer\'].statistics_,X_train[numeric].median().to_numpy())\n        imputed=num.named_steps[\'imputer\'].transform(X_train[numeric])\n        np.testing.assert_allclose(num.named_steps[\'scaler\'].mean_,imputed.mean(axis=0))\n        for col,learned in zip(categorical,pre.named_transformers_[\'cat\'].named_steps[\'onehot\'].categories_):\n            assert set(learned)==set(X_train[col].dropna().unique())\n        rows.append(dict(model=name,fit_rows=len(X_train),numeric_imputer=\'Train median verified\',scaler=\'Train mean verified\',categories=\'Train values verified\',status=\'PASS\'))\n    return pd.DataFrame(rows)\n\ndef pretest(processed_dir,report_dir,expected=None):\n    data=c.load_classification_source_data(processed_dir)\n    model_data,numeric,categorical,merge,quality=c.build_classification_dataset(**data)\n    X_train,X_valid,X_test,y_train,y_valid,y_test,features=c.split_train_validation_test(model_data,numeric,categorical,random_state=42)\n    assert not(set(X_train.index)&set(X_valid.index) or set(X_train.index)&set(X_test.index) or set(X_valid.index)&set(X_test.index))\n    models,comparison,predictions,probabilities=c.train_and_compare_on_validation(X_train=X_train,X_valid=X_valid,y_train=y_train,y_valid=y_valid,numeric_features=numeric,categorical_features=categorical,random_state=42)\n    model=c.select_validation_model(comparison)\n    thresholds=c.threshold_metrics(y_valid,probabilities[model])\n    threshold=c.choose_threshold(thresholds)\n    frozen=freeze_selection(model,threshold,comparison,thresholds,model_data,[X_train,X_valid,X_test],report_dir,expected)\n    return dict(data=data,model_data=model_data,numeric_features=numeric,categorical_features=categorical,features=features,feature_audit=c.build_feature_audit(numeric,categorical),target_distribution=c.target_distribution(model_data),merge_checks=merge,data_quality_checks=quality,split_summary=c.build_split_summary(y_train,y_valid,y_test),X_train=X_train,X_validation=X_valid,X_test=X_test,y_train=y_train,y_validation=y_valid,y_test=y_test,models=models,validation_model_comparison=comparison,validation_predictions=predictions,validation_probabilities=probabilities,selected_model_name=model,selected_threshold=threshold,threshold_metrics=thresholds,frozen=frozen,pipeline_fit_audit=fit_audit(models,X_train,numeric,categorical))\n\ndef truthful_checklist():\n    result=c.build_classification_checklist()\n    result[\'status\']=\'code_verified\'\n    result.loc[2,\'status\']=\'documented_assumption_unconfirmed_in_business\'\n    result.loc[8,\'status\']=\'frozen_first_evaluation_then_reproducibility_replay\'\n    result.loc[12:13,\'status\']=\'limitation_documented\'\n    result[\'review_scope\']=\'automated evidence; student final review is separate\'\n    return result\n\ndef complete(r,report_dir):\n    model=r[\'selected_model_name\'];threshold=r[\'selected_threshold\']\n    yp,prob,metrics=c.final_test_evaluation(r[\'models\'][model],r[\'X_test\'],r[\'y_test\'],threshold=threshold,model_name=model)\n    internal=c.create_prediction_result(r[\'y_test\'],yp,prob,model_name=model,threshold=threshold,source_index=r[\'X_test\'].index)\n    public=c.public_prediction_result(internal)\n    confusion=c.confusion_matrix_dataframe(r[\'y_test\'],yp)\n    report=c.classification_report_dataframe(r[\'y_test\'],yp)\n    validation=c.build_classification_validation(model_data=r[\'model_data\'],features=r[\'features\'],merge_checks=r[\'merge_checks\'],y_train=r[\'y_train\'],y_valid=r[\'y_validation\'],y_test=r[\'y_test\'],validation_comparison=r[\'validation_model_comparison\'],selected_model_name=model,threshold_df=r[\'threshold_metrics\'],selected_threshold=threshold,prediction_result_public=public,test_metrics=metrics)\n    # Independent counts and arithmetic do not call sklearn metric functions.\n    actual=public.actual_is_cancelled.to_numpy();pred=public.predicted_is_cancelled.to_numpy()\n    tn=int(((actual==0)&(pred==0)).sum());fp=int(((actual==0)&(pred==1)).sum())\n    fn=int(((actual==1)&(pred==0)).sum());tp=int(((actual==1)&(pred==1)).sum())\n    manual={\'accuracy\':(tp+tn)/len(actual),\'precision\':tp/(tp+fp) if tp+fp else 0.,\'recall\':tp/(tp+fn) if tp+fn else 0.,\'f1\':2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.}\n    for k,v in manual.items():assert np.isclose(metrics.iloc[0][k],v)\n    np.testing.assert_array_equal(confusion.to_numpy(),[[tn,fp],[fn,tp]])\n    np.testing.assert_array_equal(pred,(prob>=threshold).astype(int))\n    assert np.isfinite(prob).all() and ((prob>=0)&(prob<=1)).all()\n    extra=pd.DataFrame([{\'check\':key,\'value\':value,\'status\':\'PASS\'} for key,value in [\n       (\'manual_confusion_metrics\',True),(\'probability_threshold_coordinates\',True),(\'disjoint_split_indices\',True),\n       (\'freeze_event_precedes_single_test\',EVENTS==[\'selection_saved_before_test\',\'final_test_evaluated\']),\n       (\'source_scope_reconciles\',203+45+52==300),(\'pipeline_train_fit_parameters\',True)]])\n    assert extra.value.all()\n    validation=pd.concat([validation,extra],ignore_index=True)\n    validation=pd.concat([validation,pd.DataFrame([{\'check\':\'business_date_meaning_unresolved\',\'value\':45,\'status\':\'REVIEW_REQUIRED\'},{\'check\':\'prediction_time_feature_availability\',\'value\':\'assumption, historical snapshots unavailable\',\'status\':\'REVIEW_REQUIRED\'}])],ignore_index=True)\n    checklist=truthful_checklist()\n    paths=c.save_classification_outputs(model_data=r[\'model_data\'],target_dist=r[\'target_distribution\'],feature_audit=r[\'feature_audit\'],merge_checks=r[\'merge_checks\'],data_quality_checks=r[\'data_quality_checks\'],split_summary=r[\'split_summary\'],validation_comparison=r[\'validation_model_comparison\'],threshold_df=r[\'threshold_metrics\'],test_metrics=metrics,prediction_result_internal=internal,prediction_result_public=public,confusion_df=confusion,report_df=report,validation=validation,checklist=checklist,selected_model_name=model,selected_threshold=threshold,report_dir=report_dir)\n    # Original IDs and source positions stay in the temporary project.\n    for name,frame in [(\'scope_audit\',AUDIT[\'scope\']),(\'business_date_hold_internal\',AUDIT[\'held\']),(\'pipeline_fit_audit\',r[\'pipeline_fit_audit\'])]:\n        p=Path(report_dir)/f\'ch10_{name}.csv\';frame.to_csv(p,index=False,encoding=\'utf-8-sig\');paths[name]=p\n    r.update(test_metrics=metrics,prediction_result=public,prediction_result_internal=internal,confusion_matrix=confusion,classification_report=report,validation=validation,checklist=checklist,output_paths=paths)\n    out=Path(report_dir)\n    error=public.copy();error[\'error_type\']=np.select([(actual==0)&(pred==1),(actual==1)&(pred==0),(actual==1)&(pred==1)],[\'FP\',\'FN\',\'TP\'],default=\'TN\')\n    error.to_csv(out/\'ch10_error_analysis.csv\',index=False,encoding=\'utf-8-sig\')\n    paths[\'error_analysis\']=out/\'ch10_error_analysis.csv\'\n    trace=pd.DataFrame({\'sequence\':[1,2],\'event\':EVENTS});trace.to_csv(out/\'ch10_evaluation_trace.csv\',index=False)\n    paths[\'evaluation_trace\']=out/\'ch10_evaluation_trace.csv\';paths[\'selection_summary\']=out/\'ch10_selection_summary.csv\'\n    selected_val=r[\'threshold_metrics\'].loc[np.isclose(r[\'threshold_metrics\'].threshold,threshold)].iloc[0]\n    text=paths[\'report\'].read_text()\n    text += f\'\'\'\\n## 14. 분석 범위와 남은 데이터 문제\\n\\n- 원본 300주문 = 분석 203 + 업무 날짜 보류 45 + refunded 제외 52로 대사함\\n- 가입일 이전 주문은 전체 상태 56건이며 이진 대상에서는 completed 35건과 cancelled 10건을 보류함\\n- 변환 실패는 0건이며 보류가 날짜의 업무 의미를 해결한 것은 아님\\n- 공식 음수 days_since_signup Gate는 203건 분석 subset에서 충족하며 원본 문제 45건은 별도 미해결임\\n- 날짜 보류는 모델 성능을 보기 전에 정한 적격 기준이며 보류 제외로 선택편향이 생길 수 있음\\n\\n```text\\n{AUDIT[\'scope\'].to_string(index=False)}\\n```\\n\\n## 15. 예측 시점과 사전 선택 규칙\\n\\n- 상세와 결제수단이 확정됐지만 최종 취소 여부가 확정되기 전을 예측 시점으로 가정함\\n- item_count는 상세 행 수이고 total_quantity 및 order_amount는 이 시점에 확정된 quantity와 unit_price로만 계산해야 함\\n- 고객 정보와 signup_date가 당시 가용하며 age가 해당 시점 나이라고 가정함, 생년월일과 시점별 스냅샷이 없어 정밀 확인은 불가함\\n- 선택 기준은 비Dummy Validation F1, Recall, Precision, 모델명 순이며 threshold는 0.20~0.80에서 0.05 간격으로 비교하고 F1, Recall, Precision, 낮은 값 순으로 선택함\\n- FP/FN 비용은 미확정이라 교육용 F1 기준을 사용함\\n- 최초 Test 전에 선택과 입력 및 split 해시를 저장함, 이후 Notebook 및 CLI 재실행은 고정 결과 재현이며 새로운 독립 Test가 아님\\n\\n## 16. 오류 해석과 사용 판단\\n\\n- TN {tn}, FP {fp}, FN {fn}, TP {tp}이며 FP는 완료 주문을 취소로 오탐한 경우이고 FN은 실제 취소를 놓친 경우임\\n- 선택 threshold의 Validation F1 {selected_val.f1:.6f}에서 Test F1 {manual[\'f1\']:.6f}로 {manual[\'f1\']-selected_val.f1:+.6f} 변함\\n- split별 표본 구성과 적은 취소 표본이 차이에 영향을 줄 수 있으나 이 결과만으로 차이의 원인을 확정할 수 없음\\n- 예측 확률의 보정 여부가 검증되지 않았으므로 확률을 정확한 취소 위험률로 해석하지 않음\\n- 자동 검증의 PASS는 계산 및 선택 절차에 한정함, 날짜 의미와 시점별 Feature 가용성은 REVIEW_REQUIRED로 남김\n- 현재 운영 사용 보류, 시간순 평가와 고객 중복 영향, 공정성, 비용 기준 및 날짜 의미 확인이 필요함\\n- 확인한 결과는 보류 제외 표본의 예측 성능이며 주문 금액이나 고객 특성이 취소를 일으킨다는 인과 주장은 불가함\\n\\n## 17. 공개 산출물\\n\\n\'\'\'\n    for p in sorted(out.glob(\'*.csv\')):\n        if \'_internal\' not in p.name:text+=f\'- [{p.name}]({p.name})\\n\'\n    text+=\'\\n![클래스 비율](../images/step02_class_ratio.png)\\n\\n![Validation 비교](../images/step03_baseline.png)\\n\\n![Threshold 비교](../images/step05_threshold.png)\\n\\n![Test 혼동행렬](../images/step06_confusion_matrix.png)\\n\'\n    paths[\'report\'].write_text(text,encoding=\'utf8\')\n    return r\n\ndef run(processed_dir=\'data/processed\',report_dir=\'reports\',random_state=42):\n    assert random_state==42\n    expected_path=Path(__file__).parent/\'ch10_frozen_expected.json\'\n    expected=json.loads(expected_path.read_text())\n    return complete(pretest(processed_dir,report_dir,expected),report_dir)\n', 'src/ch10_frozen_expected.json': '{\n  "policy": {\n    "prediction_time": "order details and payment method fixed, before final cancellation outcome",\n    "target": "completed=0,cancelled=1; refunded excluded",\n    "eligibility": "hold binary orders before signup; no date correction; 203 eligible + 45 hold + 52 target-excluded = 300",\n    "features": "7 numeric + 3 categorical; no status,target,IDs,post-outcome data",\n    "random_state": 42,\n    "split": "stratified train60/validation20/test20",\n    "model_rule": "non-Dummy Validation F1 desc, recall desc, precision desc, model asc",\n    "threshold_rule": "0.20:0.05:0.80; Validation F1 desc, recall desc, precision desc, threshold asc",\n    "cost_assumption": "unknown FP/FN costs; educational F1 rule, not optimized business cost",\n    "usage": "hold; future-time validation and data semantics unresolved"\n  },\n  "model": "Logistic Regression",\n  "threshold": 0.6,\n  "dataset_hash": "2fe18586efabef4f1d453f2b316d1c05dd4ef494616fd89e6c661b053cf974f3",\n  "validation_hash": "ba75b85bc779c7e2bed41ddf75e854b3a1cd06e800371afbcf8f2ede6d2c79de",\n  "threshold_grid_hash": "058f5b29ada500bab5033f7000f575b5184f0754dc09fde62881e6008b93cbfa",\n  "split_hashes": {\n    "train": "2f1422ff4d2812bf59c521500ab85a7e91404554f604874925a27249c50ba186",\n    "validation": "deef58938b7c62f2d387deb42e9395762e4a8f6e46454b33b4ebde4efd4b907e",\n    "test": "3a2f76eb9086b6837a7afc2c56d2f8bd9686ba509293a6201b7bf06dad16367b"\n  }\n}'}

# Necessary source is embedded; shared course data and reference files are not written.
PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="ch10_classification_"))
for relative, content in EMBEDDED_SOURCES.items():
    destination = PROJECT_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")
for name, expected_sha in RAW_HASHES.items():
    candidates = [p / "llm-data-analysis-course-reference" / "data" / "raw" / f"{name}.csv" for p in [Path.cwd(), *Path.cwd().parents]]
    local = next((p for p in candidates if p.is_file() and hashlib.sha256(p.read_bytes()).hexdigest() == expected_sha), None)
    payload = local.read_bytes() if local else urllib.request.urlopen(f"https://raw.githubusercontent.com/GilbertMoon/llm-data-analysis-course/{REFERENCE_COMMIT}/data/raw/{name}.csv", timeout=30).read()
    assert hashlib.sha256(payload).hexdigest() == expected_sha
    dest = PROJECT_ROOT / "data" / "raw" / f"{name}.csv"
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(payload)
for module in list(sys.modules):
    if module == "src" or module.startswith("src."):
        del sys.modules[module]
sys.path.insert(0, str(PROJECT_ROOT))
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
OUTPUT_ROOT = Path.cwd().parent / "reports" / "chapter10" if Path.cwd().name == "notebooks" else Path.cwd()
PUBLIC_DIR = OUTPUT_ROOT / "reports"
IMAGE_DIR = OUTPUT_ROOT / "images"
PUBLIC_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
from scripts.prepare_ch10_data import prepare_ch10_data
prepared = prepare_ch10_data(PROJECT_ROOT / "data" / "raw", PROCESSED_DIR)
for name, raw in prepared["raw_data"].items():
    assert len(raw) == len(prepared["processed_data"][name]), f"Unexpected preprocessing exclusion: {name}"
assert prepared["relationship_checks"].invalid_count.eq(0).all()
print("Pinned raw SHA256 verified, separate temporary project, prepared without row removal")
display(prepared["relationship_checks"])


Pinned raw SHA256 verified, separate temporary project, prepared without row removal


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 2. 전처리 데이터 확인과 불러오기

이 장에서는 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (300, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (764, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


## 3. 주문 상태와 타깃 범위 확인

이번 실습은 완료 주문과 취소 주문만 비교합니다.

- `completed` → 0
- `cancelled` → 1
- `refunded`와 기타 상태 → 학습 대상에서 제외


In [3]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


,order_status,order_count
0,completed,184
1,cancelled,64
2,refunded,52


## 4. 분류 데이터와 병합 검증표 생성

공통 함수는 주문 특징 집계, 안전한 병합, 시간 순서 점검, 타깃 생성까지 수행합니다.


In [4]:
from src import classification as c
from src import ch10_review as review
(model_data, numeric_features, categorical_features, merge_checks, data_quality_checks) = c.build_classification_dataset(customers=customers, orders=orders, order_items=order_items)
target_dist = c.target_distribution(model_data)
features = numeric_features + categorical_features
feature_audit = c.build_feature_audit(numeric_features, categorical_features)
print("Modeling data", model_data.shape)
display(review.AUDIT["scope"])
display(target_dist)
display(merge_checks)
display(data_quality_checks)


Modeling data (203, 16)


,scope,order_status,order_count,customer_count,detail_rows,total_quantity,order_amount
0,eligible,cancelled,54,44,136,427.0,52759000.0
1,eligible,completed,149,84,385,1178.0,123886000.0
2,business_date_hold,cancelled,10,10,26,80.0,9422000.0
3,business_date_hold,completed,35,23,89,264.0,25104000.0
4,target_excluded,refunded,52,46,128,384.0,44439000.0


,is_cancelled,class_label,count,ratio
0,0,completed,149,0.734
1,1,cancelled,54,0.266


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status,validate_relationship
0,order_id → order_item_features,203,203,True,0,PASS,one_to_one
1,customer_id → customers,203,203,True,0,PASS,many_to_one


,check_item,count
0,binary_target_rows,203
1,completed_rows,149
2,cancelled_rows,54
3,excluded_status_rows,52
4,line_total_mismatch,0
5,negative_days_since_signup,0
6,missing_order_item_features,0
7,missing_customer_match,0
8,status_scope:completed,149
9,status_scope:cancelled,54


## 5. 데이터 누수 확인

`order_status`와 `is_cancelled`는 입력 feature에 포함되면 안 됩니다.


In [5]:
assert not (set(features) & c.FORBIDDEN_FEATURES)
assert not model_data[features].isna().any().any()
assert (model_data.days_since_signup >= 0).all()
display(feature_audit)
print("Feature missing values:", int(model_data[features].isna().sum().sum()))


,column,selected,role,reason
0,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,item_count,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,total_quantity,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,order_amount,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,days_since_signup,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
7,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
8,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
9,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정


Feature missing values: 0


## 6. train, validation, test 분리

모델과 임계값은 validation에서 선택하고, test는 최종 평가에 한 번만 사용합니다.


In [6]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)


X_train: (121, 10)
X_valid: (41, 10)
X_test: (41, 10)


,split,is_cancelled,class_label,count,ratio
0,train,0,completed,89,0.7355
1,train,1,cancelled,32,0.2645
2,validation,0,completed,30,0.7317
3,validation,1,cancelled,11,0.2683
4,test,0,completed,30,0.7317
5,test,1,cancelled,11,0.2683


## 7. 기준 모델과 학습 모델 비교

가장 많은 클래스만 예측하는 Dummy 모델을 기준으로 Logistic Regression과 Random Forest의 validation 성능을 비교합니다.


In [7]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison


,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.634146,0.400000,0.727273,0.516129
1,Random Forest,validation,0.658537,0.285714,0.181818,0.222222
2,Dummy Most Frequent,validation,0.731707,0.000000,0.000000,0.000000


## 8. validation에서 모델과 임계값 선택

Dummy 모델은 기준선이므로 실제 선택 후보에서는 제외합니다. validation의 f1-score를 우선하고 recall과 precision을 함께 확인합니다.


In [8]:
selected_model_name = c.select_validation_model(validation_comparison)
selected_model = models[selected_model_name]
threshold_df = c.threshold_metrics(y_valid, validation_probabilities[selected_model_name])
selected_threshold = c.choose_threshold(threshold_df)
expected = json.loads((PROJECT_ROOT / "src" / "ch10_frozen_expected.json").read_text())
frozen = review.freeze_selection(selected_model_name, selected_threshold, validation_comparison, threshold_df, model_data, [X_train, X_valid, X_test], REPORT_DIR, expected)
pipeline_fit_audit = review.fit_audit(models, X_train, numeric_features, categorical_features)
assert not (set(X_train.index) & set(X_valid.index) or set(X_train.index) & set(X_test.index) or set(X_valid.index) & set(X_test.index))
print("Saved frozen selection before final Test:", selected_model_name, selected_threshold)
display(threshold_df)
display(pipeline_fit_audit)


Saved frozen selection before final Test: Logistic Regression 0.6


,threshold,accuracy,precision,recall,f1
0,0.20,0.365854,0.297297,1.000000,0.458333
1,0.25,0.463415,0.322581,0.909091,0.476190
2,0.30,0.463415,0.322581,0.909091,0.476190
3,0.35,0.512195,0.333333,0.818182,0.473684
4,0.40,0.560976,0.360000,0.818182,0.500000
5,0.45,0.658537,0.428571,0.818182,0.562500
6,0.50,0.634146,0.400000,0.727273,0.516129
7,0.55,0.682927,0.444444,0.727273,0.551724
8,0.60,0.780488,0.583333,0.636364,0.608696
9,0.65,0.756098,0.555556,0.454545,0.500000


,model,fit_rows,numeric_imputer,scaler,categories,status
0,Dummy Most Frequent,121,Train median verified,Train mean verified,Train values verified,PASS
1,Logistic Regression,121,Train median verified,Train mean verified,Train values verified,PASS
2,Random Forest,121,Train median verified,Train mean verified,Train values verified,PASS


## 9. test 데이터 최종 평가

이제 선택이 끝났으므로 test 데이터에서 최종 성능을 한 번 확인합니다.


In [9]:
classification_result = dict(data={"customers":customers,"orders":orders,"order_items":order_items}, model_data=model_data, numeric_features=numeric_features, categorical_features=categorical_features, features=features, feature_audit=feature_audit, target_distribution=target_dist, merge_checks=merge_checks, data_quality_checks=data_quality_checks, split_summary=split_summary, X_train=X_train, X_validation=X_valid, X_test=X_test, y_train=y_train, y_validation=y_valid, y_test=y_test, models=models, validation_model_comparison=validation_comparison, validation_predictions=validation_predictions, validation_probabilities=validation_probabilities, selected_model_name=selected_model_name, selected_threshold=selected_threshold, threshold_metrics=threshold_df, frozen=frozen, pipeline_fit_audit=pipeline_fit_audit)
classification_result = review.complete(classification_result, REPORT_DIR)
test_metrics = classification_result["test_metrics"]
confusion_df = classification_result["confusion_matrix"]
classification_report_df = classification_result["classification_report"]
display(test_metrics)
display(confusion_df)
display(classification_report_df)


,model,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,Logistic Regression,test_final,0.6,frozen_before_test,0.682927,0.4,0.363636,0.380952


,pred_completed,pred_cancelled
actual_completed,24,6
actual_cancelled,7,4


,label,precision,recall,f1-score,support
0,completed,0.774194,0.800000,0.786885,30.000000
1,cancelled,0.400000,0.363636,0.380952,11.000000
2,accuracy,0.682927,0.682927,0.682927,0.682927
3,macro avg,0.587097,0.581818,0.583919,41.000000
4,weighted avg,0.673800,0.682927,0.677976,41.000000


혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 완료 주문을 완료로 예측 |
| False Positive | 실제 완료 주문을 취소로 잘못 예측 |
| False Negative | 실제 취소 주문을 완료로 잘못 예측 |
| True Positive | 실제 취소 주문을 취소로 예측 |


## 10. 개인정보를 제외한 예측 결과 생성

고객명, 이메일, 주소 같은 불필요한 식별정보는 결과 파일에 저장하지 않습니다.


In [10]:
prediction_result = classification_result["prediction_result"]
assert list(prediction_result.columns) == ["record_id", "actual_is_cancelled", "predicted_is_cancelled", "cancel_probability", "model", "threshold"]
display(prediction_result.head())
errors = pd.read_csv(REPORT_DIR / "ch10_error_analysis.csv")
display(errors.groupby("error_type").agg(count=("record_id","size"), probability_min=("cancel_probability","min"), probability_max=("cancel_probability","max")))
display(errors[errors.error_type.isin(["FP","FN"])].head(6))


,record_id,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,0,0,0.569824,Logistic Regression,0.6
1,test_0002,0,0,0.576583,Logistic Regression,0.6
2,test_0003,0,0,0.328096,Logistic Regression,0.6
3,test_0004,1,1,0.880826,Logistic Regression,0.6
4,test_0005,0,0,0.545805,Logistic Regression,0.6


,count,probability_min,probability_max
error_type,,,
FN,7,0.124812,0.446529
FP,6,0.601042,0.724792
TN,24,0.165958,0.597192
TP,4,0.693977,0.880826


,record_id,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold,error_type
7,test_0008,1,0,0.372579,Logistic Regression,0.6,FN
9,test_0010,1,0,0.349958,Logistic Regression,0.6,FN
11,test_0012,1,0,0.421218,Logistic Regression,0.6,FN
12,test_0013,0,1,0.682772,Logistic Regression,0.6,FP
13,test_0014,0,1,0.618451,Logistic Regression,0.6,FP
16,test_0017,1,0,0.446529,Logistic Regression,0.6,FN


## 11. LLM 코드 검토 체크리스트


In [11]:
classification_checklist = classification_result["checklist"]
display(classification_checklist)
display(classification_result["validation"])


,check_item,status,review_scope
0,completed와 cancelled만 사용해 이진 타깃을 만들었는가?,code_verified,automated evidence; student final review is se...
1,refunded 등 다른 상태를 0 클래스에 섞지 않았는가?,code_verified,automated evidence; student final review is se...
2,예측 시점과 feature 가용성 가정을 설명할 수 있는가?,documented_assumption_unconfirmed_in_business,automated evidence; student final review is se...
3,line_total = quantity × unit_price를 검증했는가?,code_verified,automated evidence; student final review is se...
4,병합에 validate를 사용하고 미매칭 0건을 확인했는가?,code_verified,automated evidence; student final review is se...
5,"order_status, target, ID, 사후 정보를 feature에서 제외했는가?",code_verified,automated evidence; student final review is se...
6,"train, validation, test를 분리했는가?",code_verified,automated evidence; student final review is se...
7,모델과 threshold는 validation에서 선택했는가?,code_verified,automated evidence; student final review is se...
8,모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?,frozen_first_evaluation_then_reproducibility_r...,automated evidence; student final review is se...
9,Dummy 기준 모델과 비교했는가?,code_verified,automated evidence; student final review is se...


,check,value,status
0,target_exactly_completed_cancelled,"[0, 1]",PASS
1,forbidden_feature_overlap,0,PASS
2,merge_rows_and_matches,True,PASS
3,all_splits_have_both_classes,True,PASS
4,model_selected_on_validation,True,PASS
5,threshold_selected_on_validation,True,PASS
6,public_prediction_identifier_columns,none,PASS
7,test_used_as_final_evaluation,True,PASS
8,manual_confusion_metrics,True,PASS
9,probability_threshold_coordinates,True,PASS


## 12. 전체 파이프라인 실행과 산출물 저장

위 과정을 한 번에 다시 실행해 CSV와 Markdown 보고서를 생성합니다.


In [12]:
# Independent processes replay the fixed protocol; they do not choose again using Test.
cli_root = Path(tempfile.mkdtemp(prefix="ch10_cli_replay_"))
shutil.copytree(PROJECT_ROOT / "src", cli_root / "src")
shutil.copytree(PROJECT_ROOT / "scripts", cli_root / "scripts")
shutil.copytree(PROJECT_ROOT / "data" / "raw", cli_root / "data" / "raw")
env = dict(os.environ)
env.pop("PYTHONPATH", None)
for script in ["prepare_ch10_data.py", "run_classification_analysis.py"]:
    done = subprocess.run([sys.executable, "scripts/" + script], cwd=cli_root, env=env, text=True, capture_output=True, check=True)
    print(script, "exit", done.returncode)
for original in sorted(PROCESSED_DIR.glob("*.csv")):
    pd.testing.assert_frame_equal(pd.read_csv(original), pd.read_csv(cli_root / "data" / "processed" / original.name))
for original in sorted(REPORT_DIR.glob("*.csv")):
    pd.testing.assert_frame_equal(pd.read_csv(original), pd.read_csv(cli_root / "reports" / original.name))
assert (REPORT_DIR / "ch10_classification_summary.md").read_text() == (cli_root / "reports" / "ch10_classification_summary.md").read_text()
print("All processed and report CSV values and Markdown match separate CLI replay")
for path in REPORT_DIR.iterdir():
    if path.suffix in [".csv", ".md"] and "_internal" not in path.name:
        shutil.copy2(path, PUBLIC_DIR / path.name)
print("Public CSV count:", len(list(PUBLIC_DIR.glob("*.csv"))))
print("Internal identifiers stay outside the submission")


prepare_ch10_data.py exit 0


run_classification_analysis.py exit 0
All processed and report CSV values and Markdown match separate CLI replay
Public CSV count: 18
Internal identifiers stay outside the submission


## 13. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 주문 취소 분류 코드를 검토해 주세요.

타깃 기준:
- completed=0
- cancelled=1
- refunded와 기타 상태는 제외

검토 항목:
1. order_status와 is_cancelled가 feature에서 제외되었는가?
2. 병합에 validate와 indicator가 사용되었는가?
3. train, validation, test가 분리되었는가?
4. 모델과 임계값은 validation에서 선택했는가?
5. test는 최종 평가에 한 번만 사용했는가?
6. Dummy 기준 모델보다 나은가?
7. accuracy, precision, recall, f1을 함께 확인했는가?
8. 예측 결과에 개인정보가 포함되지 않았는가?
9. 모델 결과를 취소 원인으로 단정하지 않았는가?

필수 수정과 권장 개선을 구분해 설명해 주세요.
```


## 14. 정리

이번 장에서는 완료 주문과 취소 주문을 명확하게 구분하고, 병합 검증, 데이터 누수 방지, train/validation/test 분리, 기준 모델 비교, validation 기반 모델·임계값 선택, 최종 test 평가를 수행했습니다.

다음 장에서는 LLM을 활용해 분석 질문과 프롬프트를 더 구체적으로 만드는 방법으로 이어집니다.


## 실행 결과 시각화

- 아래 이미지는 이번 실행 결과에서 생성했으며 실제 LLM 대화 캡처와 구분함

In [13]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':130,'savefig.dpi':160,'font.size':10})
colors=['#395B83','#E18B55','#64968F','#A27EA8']
counts=model_data.is_cancelled.value_counts().sort_index()
fig,ax=plt.subplots(figsize=(7,4))
bars=ax.bar(['Completed (0)','Cancelled (1)'],counts.values,color=colors[:2])
for bar,count in zip(bars,counts.values):ax.text(bar.get_x()+bar.get_width()/2,count+3,f'{count} ({count/len(model_data):.2%})',ha='center')
assert [b.get_height() for b in bars]==counts.tolist()
ax.set(ylim=(0,max(counts)*1.18),ylabel='Order count',title='Eligible binary orders (n=203; 45 business-date holds)')
fig.tight_layout();fig.savefig(IMAGE_DIR/'step02_class_ratio.png');plt.close(fig)
metrics=['accuracy','precision','recall','f1']
table=validation_comparison.set_index('model').loc[['Dummy Most Frequent','Logistic Regression','Random Forest'],metrics]
fig,ax=plt.subplots(figsize=(9,4))
table.plot.bar(ax=ax,color=colors,rot=0)
ax.set(ylim=(0,1),ylabel='Validation score',xlabel='',title='Model comparison at default threshold 0.50')
ax.legend(loc='upper right',ncol=4,fontsize=8);fig.tight_layout();fig.savefig(IMAGE_DIR/'step03_baseline.png');plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4.5))
for metric,color in zip(['precision','recall','f1'],colors):
 line,=ax.plot(threshold_df.threshold,threshold_df[metric],marker='o',label=metric,color=color)
 np.testing.assert_allclose(line.get_ydata(),threshold_df[metric])
ax.axvline(selected_threshold,color='#444',ls='--',label=f'Frozen threshold {selected_threshold:.2f}')
ax.set(ylim=(0,1.06),xlabel='Threshold',ylabel='Validation score',title=f'{selected_model_name}: Validation threshold comparison')
ax.legend();ax.grid(alpha=.2);fig.tight_layout();fig.savefig(IMAGE_DIR/'step05_threshold.png');plt.close(fig)
fig,ax=plt.subplots(figsize=(6,4.5))
mat=confusion_df.to_numpy();im=ax.imshow(mat,cmap='Blues',vmin=0)
for i in range(2):
 for j in range(2):ax.text(j,i,f'{[["TN","FP"],["FN","TP"]][i][j]} = {mat[i,j]}',ha='center',va='center',fontsize=16,color='white' if mat[i,j]>mat.max()/2 else '#222')
ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Completed (0)','Cancelled (1)'],yticklabels=['Completed (0)','Cancelled (1)'],xlabel='Predicted',ylabel='Actual',title=f'Final Test n={mat.sum()} | {selected_model_name} @ {selected_threshold:.2f}')
assert mat.sum()==len(y_test)
fig.colorbar(im,ax=ax,label='Order count');fig.tight_layout();fig.savefig(IMAGE_DIR/'step06_confusion_matrix.png');plt.close(fig)
print('Four plots generated; class heights, threshold coordinates and confusion orientation verified')


Four plots generated; class heights, threshold coordinates and confusion orientation verified


# Chapter 10 답안 양식. 분류 분석으로 주문 취소 여부 예측하기

> 이 내용을 `chapter10.ipynb`의 Markdown 셀로 작성합니다. 실제 실행 결과를 근거로 작성하고, 예측 패턴을 원인처럼 단정하지 않습니다.

## 제출 정보
이름: 장태원  
GitHub ID: pmsky1  
작성일: 2026-11-16  
- 최종 Notebook URL: https://github.com/pmsky1/llm-data-analysis-study/blob/main/chapter10/chapter10.ipynb

## 1. 분류 문제 정의
- 타깃: is_cancelled
- 0의 의미: completed
- 1의 의미: cancelled
- 제외한 상태: refunded 52건, 그 외 상태 0건
- 예측 단위: 주문 1건
- 예측 시점: 주문 상세와 결제수단이 확정되고 최종 취소 여부는 아직 확정되지 않은 시점
- 예측 시점에 사용할 수 있는 정보: 당시 고객 정보, 가입일, 주문일, 결제수단 및 확정된 상세 수량과 단가
- 예측 이후에만 알 수 있어 제외한 정보: 최종 order_status, is_cancelled, 취소 사유와 취소 일시

### 나의 해석과 판단
왜 이 타깃 범위와 예측 시점을 사용했는지 작성하세요.

- 완료와 취소를 구분하려고 refunded를 제외함
- 이진 대상 248건 중 가입일 이전 45건은 날짜 의미 확인이 필요해 보류하고 203건을 분석하며, 203 + 45 + 52 = 300으로 대사함

---

## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature

- age, item_count, total_quantity, order_amount, order_month, order_dayofweek, days_since_signup

### 사용한 범주형 Feature

- gender, city, payment_method

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
취소 이후 정보
```

### 나의 해석과 판단
파일에 컬럼이 존재하는 것과 예측 시점에 사용할 수 있는 것이 왜 다른지 작성하세요.

- 저장된 최종값은 주문 당시 값과 다를 수 있어 상세와 고객 정보의 시점별 스냅샷 확인이 필요함
- signup_date가 당시 가용하다고 가정하며 age는 생년월일이 없어 주문 시점 나이를 정밀하게 확인할 수 없음

---

## 3. 주문 단위 특징과 병합 검증

### 계산 관계 확인

```text
line_total = quantity × unit_price
```

- 계산 불일치 건수: 상세 764행에서 0건
- 주문 특징 `order_id` 중복 건수: 0건

### 병합 검증

| 병합 | 관계 | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정 |
| --- | --- | ---: | ---: | ---: | --- |
| 주문 ↔ 주문 특징 | one_to_one | 203 | 203 | 0 | PASS |
| 주문 ↔ 고객 | many_to_one | 203 | 203 | 0 | PASS |

### 결과 관찰

- 원본 PK 결측과 중복 및 FK 불일치는 0건이며 item_count는 상품 종류 수가 아닌 상세 행 수임
- 날짜 변환 실패는 0건이지만 가입일 이전은 전체 56건, 이진 대상 45건으로 별도 보류함

### 나의 해석과 판단
미매칭을 원인 확인 없이 0으로 채우면 안 되는 이유를 작성하세요.

- 누락된 관계를 실제 주문량이나 금액 0으로 바꾸면 데이터 오류가 분석값으로 섞이므로 연결 원인부터 확인해야 함

---

## 4. 클래스 비율

| 클래스 | 의미 | 개수 | 비율 |
| ---: | --- | ---: | ---: |
| 0 | completed | 149 | 73.40% |
| 1 | cancelled | 54 | 26.60% |

![클래스 비율](images/step02_class_ratio.png)

### 결과 관찰

- 분석 203건 중 completed가 다수이며 날짜 보류 45건은 이 비율에 포함하지 않음

### 나의 해석과 판단
클래스 불균형이 accuracy 해석에 어떤 영향을 주는지 작성하세요.

- 모두 completed로 예측해도 Validation Accuracy가 0.731707이므로 정확도만으로 취소 탐지 성능을 판단하기 어려움

---

## 5. Dummy Baseline과 Validation 모델 비교

| 모델 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Dummy Most Frequent | 0.731707 | 0.000000 | 0.000000 | 0.000000 |
| Logistic Regression | 0.634146 | 0.400000 | 0.727273 | 0.516129 |
| Random Forest | 0.658537 | 0.285714 | 0.181818 | 0.222222 |

![Baseline 비교](images/step03_baseline.png)

- Validation에서 선택한 모델: Logistic Regression

### 결과 관찰

- 기본 threshold 0.50의 후보 비교에서 Logistic Regression의 F1이 가장 높음

### 나의 해석과 판단
선택한 모델이 Dummy baseline보다 어떤 점에서 나아졌는지 작성하세요.

- Dummy의 취소 Recall과 F1은 0이며 Logistic Regression은 Recall 0.727273과 F1 0.516129로 취소를 일부 탐지함

### 한계와 추가 확인 사항

- Accuracy는 Dummy보다 낮고 Validation 취소가 11건뿐이므로 이 결과만으로 운영 우수성을 확정할 수 없음

---

## 6. Train / Validation / Test 역할

| Split | 역할 | 클래스 0 건수 | 클래스 1 건수 |
| --- | --- | ---: | ---: |
| Train | 모델 학습 | 89 | 32 |
| Validation | 모델·Threshold 선택 | 30 | 11 |
| Test | 선택 이후 최종 평가 | 30 | 11 |

### 나의 해석과 판단
왜 Test를 모델 또는 Threshold 선택에 사용하면 안 되는지 작성하세요.

- Test를 선택에 사용하면 최종 평가가 선택 과정에 맞춰져 새로운 주문에 대한 성능을 과대평가할 수 있음

### 교육용 random split의 한계
실제 미래 주문 예측 운영 전에 추가해야 할 평가를 작성하세요.

- random split은 시간 순서와 같은 고객의 중복을 막지 않으므로 미래 기간 분할과 고객 단위 분리 평가가 필요함

---

## 7. Validation 성능과 우선 지표

- Accuracy: 0.780488
- Precision: 0.583333
- Recall: 0.636364
- F1: 0.608696

### 내가 가장 중요하게 본 지표

- 선택 threshold 0.60의 Validation F1을 우선하며 5번의 기본 threshold 0.50 지표와 구분함

### 그 이유

- 실제 FP/FN 비용이 없어 교육용으로 Precision과 Recall을 함께 보는 F1을 사전에 정함
- 동률이면 Recall, Precision 순이며 비용을 최적화한 기준은 아님

> 실제 FP/FN 비용 정보가 없다면 가정임을 명시하세요.

---

## 8. Threshold 비교

| Threshold | Precision | Recall | F1 | 예상 특징 |
| ---: | ---: | ---: | ---: | --- |
| 0.50 | 0.400000 | 0.727273 | 0.516129 | 기본 기준 |
| 0.60 | 0.583333 | 0.636364 | 0.608696 | 비교한 범위에서 F1 최대 |
| 0.70 | 0.800000 | 0.363636 | 0.500000 | Precision 증가, Recall 감소 |

![Threshold 비교](images/step05_threshold.png)

### 내가 선택한 Threshold

- 0.60

### 선택 이유

- 0.20~0.80을 0.05 간격으로 비교한 13개 값 중 Validation F1이 가장 높음
- 동률 처리 순서도 F1, Recall, Precision, 낮은 threshold로 Test 전에 고정함

### 나의 해석과 판단
Threshold를 Final Test가 아니라 Validation에서 정해야 하는 이유를 작성하세요.

- threshold도 예측값을 바꾸는 선택이므로 Test에 맞춰 조정하면 독립 평가가 되지 않음

---

## 9. FP/FN 해석

- FP 의미: 실제 completed를 cancelled로 예측한 6건
- FN 의미: 실제 cancelled를 completed로 예측한 7건
- 현재 목적에서 더 부담스러울 수 있는 오류: 취소 위험을 미리 찾아 검토한다는 목적에서는 FN을 우선 주의해야 함, 다만 실제 비용의 우열은 확인할 수 없어 기존 F1 선택 기준은 유지함
- 그 판단에 추가로 필요한 업무 정보: 불필요한 고객 연락 비용, 실제 취소 손실, 취소 예방 개입의 효과

![Confusion Matrix](images/step06_confusion_matrix.png)

### 업무·분석적 의미

- TN 24건과 TP 4건이며 실제 취소 11건 중 7건을 놓쳐 자동 개입에 바로 쓰기 어려움
- FP 6건은 불필요한 고객 연락, 상담원 검토, 예방 혜택 제공 비용을 만들 수 있으나 실제 발생 여부와 금액은 미확인임
- FN 7건은 취소 위험에 대한 사전 확인이나 예방 기회를 놓친 경우이며, 개입하면 취소를 막을 수 있다는 뜻은 아님
- 비용 비교에는 연락 및 검토 단가, 주문별 이익과 취소 손실, 예방 성공률, 고객 불편 및 처리 가능 인력이 추가로 필요함

### 한계와 추가 확인 사항

- 확률 보정을 확인하지 않아 cancel_probability를 정확한 취소 위험률로 단정할 수 없음

---

## 10. Final Test

- 고정한 모델: Logistic Regression
- 고정한 Threshold: 0.60
- Accuracy: 0.682927
- Precision: 0.400000
- Recall: 0.363636
- F1: 0.380952
- Validation 대비 변화: 같은 threshold의 F1 0.608696에서 0.380952로 0.227743 감소

### 결과 관찰

- 모델과 threshold 및 입력과 분할 해시를 저장한 뒤 Test를 평가했으며 결과를 보고 선택을 변경하지 않음
- 이후 새 커널과 별도 CLI 실행은 같은 고정 결과의 재현이며 새로운 독립 Test로 세지 않음

### 나의 해석과 판단
Final Test 결과를 보고 모델의 일반화 가능성을 어떻게 판단했는지 작성하세요.

- Validation보다 낮은 성능으로 안정적인 일반화를 확인하기 어려워 사용을 보류함
- 작은 취소 표본과 split 구성이 영향을 줄 수 있으나 차이의 원인을 확정할 수 없음

### 선택 보호 확인
- [x] Test 결과를 보기 전에 모델을 고정했습니다.
- [x] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [x] Test 결과를 본 뒤 같은 Test를 이용해 선택을 바꾸지 않았습니다.

---

## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰

- 공개 prediction은 41행과 지정된 6개 컬럼이며 원본 ID와 source_index가 없음
- 원본과 연결 가능한 내부 결과 및 날짜 보류 상세는 제출 폴더 밖에 보존함

### 나의 해석과 판단
모델 오류 분석용 내부 자료와 공개 결과를 왜 분리해야 하는지 작성하세요.

- 내부 자료는 원인을 추적할 연결 키가 필요하지만 공개 결과는 예측 확인에 필요한 값만 제공해 원본 고객과 주문 노출을 줄여야 함

---

## 12. Validation Evidence

다음 Evidence 중 확인한 항목에 체크합니다.

- [x] `ch10_target_distribution.csv`
- [x] `ch10_feature_audit.csv`
- [x] `ch10_merge_checks.csv`
- [x] `ch10_data_quality_checks.csv`
- [x] `ch10_split_summary.csv`
- [x] `ch10_validation_model_comparison.csv`
- [x] `ch10_validation_threshold_metrics.csv`
- [x] `ch10_test_metrics.csv`
- [x] `ch10_confusion_matrix.csv`
- [x] `ch10_classification_validation.csv`

### 자동 Validation 결과

- 계산 및 선택 절차 14항목은 PASS이며 날짜 의미와 예측 시점 Feature 가용성 2항목은 REVIEW_REQUIRED임
- 별도 CLI 2개가 성공했고 공개 및 내부 CSV 전체와 요약 보고서가 Notebook 결과와 일치함

### 자동 PASS가 의미하지 않는 것
운영 적합성, 공정성, 시간에 따른 성능 변화 등 자동 검증만으로 판단할 수 없는 내용을 작성하세요.

- PASS는 계산과 절차 검증이며 날짜 문제 해결이나 운영 적합성, 공정성 및 미래 성능을 보장하지 않음

---

## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 사용 보류

### 판단 근거
1. completed 73.40%의 불균형에서 Dummy Validation F1은 0이며 선택 모델은 취소를 일부 탐지함, threshold 0.60은 Validation F1 최대값으로 선택했지만 Test Precision 0.400000, Recall 0.363636, F1 0.380952로 취소 11건 중 7건을 놓침
2. 날짜 보류 45건과 Feature 시점 가정이 미해결임
3. 미래 기간 성능과 FP/FN 비용을 확인하지 못함, 공개 예측의 원본 식별자는 제거했지만 이것이 운영 적합성을 보장하지는 않음

### 현재 모델의 가장 큰 위험

- 적은 평가 표본과 날짜 보류로 제한된 결과를 전체 미래 주문에 일반화하는 위험이 있음

### 다음 개선 우선순위
1. 가입일과 주문일 의미 및 당시 Feature 가용성 확인
2. 새 기간의 데이터로 시간순 평가와 고객 중복 영향 점검
3. FP/FN 비용과 확률 보정 및 공정성 확인

### 예측과 인과 구분
현재 모델에서 확인한 것은 어떤 **예측 패턴**이며, 어떤 **원인 주장**은 아직 할 수 없는지 작성하세요.

- 보류를 제외한 표본에서 취소를 일부 구분하는 예측 성능을 확인함
- 고객 나이나 주문 금액이 취소를 일으킨다는 원인 주장은 할 수 없음

---

## LLM 활용과 실제 검증 기록

- 사용한 AI 도구: ChatGPT
- 이번 검토 요청: "과제 최종 점검해줘 누락된건 없는지 등등 꼼꼼히 체크해줘"
- 제공한 맥락: Chapter 10 요구사항, 공식 실습 가이드와 답안 템플릿, 실제 Notebook 및 공개 reports, 이전 Chapter의 감점 피드백
- 아래 내용은 실제 파일 검토 결과의 요약이며 별도 대화 화면 캡처를 대신한 원문 인용은 아님

| 검토 제안 | 실제 확인 근거 | 처리와 이유 |
| --- | --- | --- |
| FP/FN의 업무 의미를 더 구체적으로 설명 | Test 혼동행렬 FP 6, FN 7, 실제 취소 11건 확인 | 채택: 가능한 연락 비용과 놓치는 예방 기회를 구분하고 실제 비용은 미확인으로 설명함 |
| 최종 판단에 Baseline과 threshold 근거도 함께 제시 | Dummy Validation F1 0, Logistic Regression 기본 F1 0.516129, threshold 0.60의 F1 0.608696 확인 | 채택: Accuracy만으로 판단하지 않고 Test F1 0.380952와 함께 설명함 |
| 실행 성공을 전체 품질 해결로 해석하지 않기 | 자동 검증 14 PASS와 2 REVIEW_REQUIRED, 원본 300 = 분석 203 + 날짜 보류 45 + refunded 52 대사 | 수정: 계산 검증 통과와 업무상 날짜 및 시점 정보 미확인을 구분함 |
| 현재 모델의 운영 사용 여부 검토 | Test Recall 4/11, 날짜 의미 및 당시 Feature 가용성 미확인 | 보류: 운영 사용은 보류하며 추가 시간순 평가와 비용 확인이 필요함 |

- 검토 후 모델, threshold, 데이터 분할 및 최초 Test 예측값은 변경하지 않음

---

## 최종 체크

- [x] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [x] `refunded`와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다.
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 해석했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] Validation Evidence를 확인했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [ ] 최종 Notebook URL을 제출합니다.


## 공개 결과 파일

- [ch10_classification_checklist.csv](reports/ch10_classification_checklist.csv)
- [ch10_classification_predictions.csv](reports/ch10_classification_predictions.csv)
- [ch10_classification_report.csv](reports/ch10_classification_report.csv)
- [ch10_classification_summary.md](reports/ch10_classification_summary.md)
- [ch10_classification_validation.csv](reports/ch10_classification_validation.csv)
- [ch10_confusion_matrix.csv](reports/ch10_confusion_matrix.csv)
- [ch10_data_quality_checks.csv](reports/ch10_data_quality_checks.csv)
- [ch10_error_analysis.csv](reports/ch10_error_analysis.csv)
- [ch10_evaluation_trace.csv](reports/ch10_evaluation_trace.csv)
- [ch10_feature_audit.csv](reports/ch10_feature_audit.csv)
- [ch10_merge_checks.csv](reports/ch10_merge_checks.csv)
- [ch10_pipeline_fit_audit.csv](reports/ch10_pipeline_fit_audit.csv)
- [ch10_scope_audit.csv](reports/ch10_scope_audit.csv)
- [ch10_selection_summary.csv](reports/ch10_selection_summary.csv)
- [ch10_split_summary.csv](reports/ch10_split_summary.csv)
- [ch10_target_distribution.csv](reports/ch10_target_distribution.csv)
- [ch10_test_metrics.csv](reports/ch10_test_metrics.csv)
- [ch10_validation_model_comparison.csv](reports/ch10_validation_model_comparison.csv)
- [ch10_validation_threshold_metrics.csv](reports/ch10_validation_threshold_metrics.csv)

## LLM 활용과 실제 확인

- 사용한 AI 도구: ChatGPT
- 요청 요약: 공식 분류 과제의 실행 오류를 확인하고 Target, 병합, 데이터 누수, Validation 선택 및 공개 결과를 검증함
- 코드 실행으로 src import 오류와 가입일 이전 45건 Gate 중단을 재현하고, import 초기화 보완 및 별도 날짜 보류를 적용함
- 자동 코드 검증 결과와 사람의 최종 답안 검토는 구분함
- [ ] 실제 LLM 대화 화면에서 Prompt와 응답을 확인함
- [ ] 최종 답안을 본인이 검토함

## 공식 실행 오류 확인

- 공식 스크립트를 프로젝트 루트에서 실행할 때 src를 찾지 못하는 오류를 확인해 내장 사본에 프로젝트 루트 import 초기화를 추가함
- import 후에는 가입일 이전 이진 주문 45건 때문에 공식 품질 Gate가 중단됨, 날짜를 고치거나 검사를 끄지 않고 사전 정의한 보류를 적용함
- 다른 학생이 말한 오류와 같은지는 해당 오류 화면이 없어 확인하지 못함
